In [1]:
import os, csv, json, time, base64, html, subprocess, tempfile, requests, threading, traceback, re, uuid, getpass, secrets as _pysecrets
from urllib.parse import urlparse, parse_qs
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from typing import Literal, Optional
from pydantic import BaseModel, Field
from IPython.display import display, HTML, JSON
from google.colab import output, userdata
from google import genai
from google.genai import types, errors

# ============================================================
# 1. إعداد العميل والثوابت الأساسية (RELAY)
# ============================================================
try:
    API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
except Exception:
    API_KEY = ""
if not API_KEY:
    API_KEY = getpass.getpass("Enter your GEMINI_API_KEY to launch RELAY: ").strip()
if not API_KEY:
    raise ValueError("GEMINI_API_KEY is required to run RELAY.")

client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=10000))
CAIRO_TZ = ZoneInfo("Africa/Cairo")

APP_PROMPT_VERSION = "2026-11-13.7-RELAY-GENDER-FINAL"
LOG_NATIONAL_ID_MODE = "masked"
RETENTION_DAYS = 0
DRAFT_TTL_SEC = 2 * 3600
TRIAGE_LOCKED_MODEL = None
APP_TOKEN = _pysecrets.token_urlsafe(16)
PARAMEDIC_ACCOUNTS = {
    "EMS001": {"pin": "2580", "name": "Paramedic 1"},
    "EMS002": {"pin": "3690", "name": "Paramedic 2"},
}

# ============================================================
# 2. سلسلة النماذج الأسرع (Ultra-Low Latency Model Chain)
# ============================================================
PREFERRED_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-2.5-flash-lite",
    "gemini-flash-lite-latest",
    "gemini-2.5-flash",
    "gemini-flash-latest",
    "gemini-3-flash-preview",
]
_EXCLUDED_KEYWORDS = ("image", "tts", "live", "audio", "embedding", "robotics", "computer-use")

def build_model_chain():
    available = []
    try:
        for m in client.models.list():
            name = (m.name or "").replace("models/", "")
            actions = getattr(m, "supported_actions", None) or []
            if "generateContent" in actions and "flash" in name and not any(k in name for k in _EXCLUDED_KEYWORDS):
                available.append(name)
    except Exception:
        pass
    if not available:
        return PREFERRED_MODELS[:]
    chain = [m for m in PREFERRED_MODELS if m in available]
    chain += [m for m in available if m not in chain][:3]
    return chain or PREFERRED_MODELS[:]

MODEL_CHAIN = build_model_chain()
print("RELAY — Bridging the Gap Before Arrival | سلسلة النماذج المعتمدة:", " ← ".join(MODEL_CHAIN))

RETRIABLE_CODES = {408, 429, 500, 502, 503, 504}
FATAL_CODES = {401, 403}

def _thinking_cfg(model):
    return None if "pro" in model else types.ThinkingConfig(thinking_budget=0)

def generate_with_fallback(contents, make_config, parse, budget_sec=20, rounds=2, models=None):
    start = time.time()
    last_err = "لا توجد محاولات"
    backoff = 0.5
    for attempt in range(rounds):
        any_retriable = False
        for model in (models or MODEL_CHAIN):
            if time.time() - start > budget_sec:
                raise RuntimeError(f"انتهت المهلة الكلية ({budget_sec}ث). آخر خطأ: {last_err}")
            for use_thinking in (True, False):
                try:
                    resp = client.models.generate_content(
                        model=model, contents=contents, config=make_config(model, use_thinking))
                    return parse(resp), model
                except errors.APIError as e:
                    code = getattr(e, "code", None)
                    last_err = f"{model}: [{code}] {getattr(e, 'message', e)}"
                    if code == 400 and use_thinking:
                        continue
                    if code in FATAL_CODES:
                        raise RuntimeError(last_err)
                    if code in RETRIABLE_CODES:
                        any_retriable = True
                    break
                except Exception as e:
                    last_err = f"{model}: {e}"
                    any_retriable = True
                    break
        if not any_retriable:
            break
        if attempt < rounds - 1:
            time.sleep(backoff)
            backoff *= 1.5
    raise RuntimeError(f"تعذر الحصول على رد من أي نموذج. آخر خطأ: {last_err}")

# ============================================================
# 3. ملف السجلات (CSV) وواجهة بيانات المستشفى الحية
# ============================================================
CSV_FILE = "relay_triage_log.csv"
CSV_HEADER = ["التوقيت", "رقم المريض", "الاسم", "العمر", "النوع", "الرقم القومي", "الشكوى", "الفلاج", "نسبة الخطورة", "القسم والتجهيز",
              "الضغط", "الأكسجين", "نموذج الذكاء الاصطناعي", "اقتراح الذكاء الأصلي", "عُدّل بشرياً", "المراجع",
              "سبب التعديل", "إصدار التعليمات", "معدل التنفس", "الحرارة", "الأدوية الحالية", "النبض", "وقت الوصول (د)", "AVPU", "GCS", "MAP (مم زئبق)"]

def _mask_id(v):
    v = str(v or "").strip()
    if LOG_NATIONAL_ID_MODE == "none" or v in ("", "-"):
        return "-"
    if LOG_NATIONAL_ID_MODE == "masked" and v.isdigit() and len(v) >= 6:
        return "*" * (len(v) - 4) + v[-4:]
    return v

def _ensure_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='', encoding='utf-8-sig') as f:
            csv.writer(f).writerow(CSV_HEADER)
        return
    with open(CSV_FILE, mode='r', encoding='utf-8-sig') as f:
        rows = list(csv.reader(f))
    if not rows:
        rows = [CSV_HEADER]
    elif len(rows[0]) < len(CSV_HEADER):
        rows[0] = CSV_HEADER
    else:
        return
    with open(CSV_FILE, mode='w', newline='', encoding='utf-8-sig') as f:
        csv.writer(f).writerows(rows)

def _purge_old_rows():
    if RETENTION_DAYS <= 0:
        return
    try:
        with open(CSV_FILE, mode='r', encoding='utf-8-sig') as f:
            rows = list(csv.reader(f))
        cutoff = datetime.now(CAIRO_TZ).replace(tzinfo=None) - timedelta(days=RETENTION_DAYS)
        keep = rows[:1]
        for r in rows[1:]:
            try:
                if datetime.strptime(r[0], "%Y-%m-%d %H:%M:%S") < cutoff:
                    continue
            except Exception:
                pass
            keep.append(r)
        if len(keep) != len(rows):
            with open(CSV_FILE, mode='w', newline='', encoding='utf-8-sig') as f:
                csv.writer(f).writerows(keep)
    except Exception:
        pass

_ensure_csv()
_purge_old_rows()

def _safe_cell(v):
    s = str(v)
    return "'" + s if s[:1] in ("=", "+", "-", "@") else s

def _generate_patient_id():
    """يولّد رقماً تعريفياً فريداً وقابلاً للتتبع للمريض بصيغة PT-YYYYMMDD-NNN-XXX."""
    date_part = datetime.now(CAIRO_TZ).strftime("%Y%m%d")
    seq = _case_number() + 1
    rand_suffix = uuid.uuid4().hex[:3].upper()
    return f"PT-{date_part}-{seq:03d}-{rand_suffix}"

def _format_gender_ar(g):
    g_u = str(g or "").strip().lower()
    if g_u in ("male", "m", "ذكر"):
        return "ذكر (Male)"
    if g_u in ("female", "f", "أنثى"):
        return "أنثى (Female)"
    return "-"

def get_saved_rows_html():
    empty = '<tr><td colspan="13" style="text-align:center;color:#64748b;">لا توجد حالات مؤكدة ومسجلة بعد.</td></tr>'
    if not os.path.exists(CSV_FILE):
        return empty
    with open(CSV_FILE, mode='r', encoding='utf-8-sig') as f:
        rows = [r for r in csv.reader(f)][1:]
    rows = [r for r in rows if len(r) >= 9]
    if not rows:
        return empty
    out = ""
    e = html.escape
    for r in reversed(rows):
        pid = r[1]
        name = r[2]
        age = r[3]
        gender = r[4] if r[4] else "-"
        nid = r[5]
        summary = r[6]
        flag_ar = r[7]
        score = r[8]
        color = "#d9534f" if flag_ar == "أحمر" else ("#e67e22" if flag_ar == "برتقالي" else "#28a745")
        dept = r[9].split(" | ")[0] if len(r) > 9 else ""
        bp = r[10] if len(r) > 10 and r[10] not in ("", "-") else ""
        sp = r[11] if len(r) > 11 and r[11] not in ("", "-") else ""
        rr = r[18] if len(r) > 18 and r[18] not in ("", "-") else ""
        tp = r[19] if len(r) > 19 and r[19] not in ("", "-") else ""
        hr = r[21] if len(r) > 21 and r[21] not in ("", "-") else ""
        eta = r[22] if len(r) > 22 and r[22] not in ("", "-") else ""
        avpu = r[23] if len(r) > 23 and r[23] not in ("", "-") else ""
        gcs = r[24] if len(r) > 24 and r[24] not in ("", "-") else ""
        map_v = r[25] if len(r) > 25 and r[25] not in ("", "-") else ""
        vit_txt = " | ".join(x for x in (
            f"{bp} mmHg" if bp else "", f"MAP {map_v} mmHg" if map_v else "", f"HR {hr}" if hr else "",
            f"SpO₂ {sp}%" if sp else "", f"RR {rr}" if rr else "", f"{tp}°C" if tp else "",
            f"ETA {eta}m" if eta else "", f"AVPU {avpu}" if avpu else "", f"GCS {gcs}" if gcs else ""
        ) if x) or "-"
        model = r[12] if len(r) > 12 and r[12] else "سجل قديم"
        edited = len(r) > 14 and r[14].startswith("نعم")
        reviewer = (r[15] if len(r) > 15 and r[15] else "-") + (" ✎" if edited else "")
        out += (
            f"<tr><td style='white-space:nowrap'>{e(r[0])}</td>"
            f"<td style='white-space:nowrap;font-family:monospace;font-weight:bold;color:#0284c7;'>{e(pid)}</td>"
            f"<td><b>{e(name)}</b></td><td>{e(age)}</td><td style='white-space:nowrap'>{e(gender)}</td>"
            f"<td>{e(nid)}</td><td>{e(summary)}</td>"
            f"<td style='color:white;background-color:{color};font-weight:bold;text-align:center;'>{e(flag_ar)}</td>"
            f"<td style='font-weight:bold;text-align:center;white-space:nowrap'>{e(score)} / 10</td>"
            f"<td>{e(dept)}</td><td style='white-space:nowrap'>{e(vit_txt)}</td>"
            f"<td style='font-size:11px'>{e(model)}</td><td style='white-space:nowrap'>{e(reviewer)}</td></tr>"
        )
    return out

def get_er_cases_v1():
    """تُرجع جميع الحالات المؤكدة بصيغة JSON منظمة لشاشة استقبال الطوارئ (RELAY ER Dashboard)."""
    if not os.path.exists(CSV_FILE):
        return JSON({"cases": []})
    with open(CSV_FILE, mode='r', encoding='utf-8-sig') as f:
        rows = [r for r in csv.reader(f)][1:]
    rows = [r for r in rows if len(r) >= 9]
    cases = []
    total_rows = len(rows)
    for idx, r in enumerate(reversed(rows)):
        case_no = total_rows - idx
        flag_ar = r[7]
        tier = "RED" if flag_ar == "أحمر" else ("ORANGE" if flag_ar == "برتقالي" else "GREEN")
        dep_parts = (r[9] if len(r) > 9 else "").split(" | ")
        department = dep_parts[0] if dep_parts else "-"
        prep_sections = [p.strip() for p in dep_parts[1:] if p.strip()]
        cases.append({
            "case_no": case_no,
            "patient_id": r[1],
            "timestamp": r[0],
            "name": r[2],
            "age": r[3],
            "gender": r[4] if r[4] else "-",
            "national_id": r[5],
            "complaint": r[6],
            "tier": tier,
            "flag_ar": flag_ar,
            "score": r[8],
            "department": department,
            "prep_sections": prep_sections,
            "bp": r[10] if len(r) > 10 and r[10] else "-",
            "spo2": r[11] if len(r) > 11 and r[11] else "-",
            "model": r[12] if len(r) > 12 and r[12] else "-",
            "edited": r[14] if len(r) > 14 and r[14] else "لا",
            "reviewer": r[15] if len(r) > 15 and r[15] else "-",
            "rr": r[18] if len(r) > 18 and r[18] else "-",
            "temp": r[19] if len(r) > 19 and r[19] else "-",
            "meds": r[20] if len(r) > 20 and r[20] else "-",
            "hr": r[21] if len(r) > 21 and r[21] else "-",
            "eta": r[22] if len(r) > 22 and r[22] else "-",
            "avpu": r[23] if len(r) > 23 and r[23] else "-",
            "gcs": r[24] if len(r) > 24 and r[24] else "-",
            "map": r[25] if len(r) > 25 and r[25] else "-",
        })
    return JSON({"cases": cases})

# ============================================================
# 4. المخطط السريري (ESI) + بروتوكولات الأطفال (PALS) + الوعي + MAP
# ============================================================
class TriageDecision(BaseModel):
    tier: Literal["RED", "ORANGE", "GREEN"] = Field(description="RED, ORANGE or GREEN only")
    risk_score: int = Field(description="Integer 1-10 reflecting true clinical severity")
    clinical_summary: str = Field(description="ملخص طبي سريري موجز للحالة باللغة العربية")
    target_department: str = Field(description="قسم واحد يُنسخ حرفياً من قائمة الأقسام المعتمدة")
    immediate_actions: list[str] = Field(description="عناصر تُنسخ حرفياً من القائمة المعتمدة")
    staff_to_alert: list[str] = Field(description="عناصر تُنسخ حرفياً من القائمة المعتمدة")
    equipment: list[str] = Field(description="عناصر تُنسخ حرفياً من القائمة المعتمدة")
    tests_and_imaging: list[str] = Field(description="عناصر تُنسخ حرفياً من القائمة المعتمدة")
    medications_to_prepare: list[str] = Field(description="عناصر تُنسخ حرفياً من القائمة المعتمدة")
    precautions: list[str] = Field(description="عناصر تُنسخ حرفياً من القائمة المعتمدة")
    other_note: str = Field(description="ملاحظة حرة قصيرة جداً للحالات الاستثنائية فقط")

DEPT_CATALOG = [
    "الطوارئ - غرفة الإنعاش والحالات الحرجة", "الطوارئ - الملاحظة والتدخل السريع",
    "الطوارئ - العيادة السريعة (حالات بسيطة)", "الصدمات والإصابات", "القلب والأوعية الدموية",
    "الجهاز التنفسي", "السكتة الدماغية والأعصاب", "طوارئ الأطفال", "النساء والتوليد",
    "الجراحة العامة", "العظام والكسور", "السموم", "الحروق",
]
DEPT_OTHER = "أخرى (حدّدها يدوياً)"
DEPT_DEFAULT_BY_TIER = {"RED": DEPT_CATALOG[0], "ORANGE": DEPT_CATALOG[1], "GREEN": DEPT_CATALOG[2]}

PREP_LABELS = [
    ("immediate_actions", "إجراءات فورية"), ("staff_to_alert", "إخطار"), ("equipment", "معدات"),
    ("tests_and_imaging", "فحوصات"), ("medications_to_prepare", "أدوية"), ("precautions", "احتياطات"),
]
PREP_UI = {
    "immediate_actions": ("🚑 إجراءات فورية", ""),
    "staff_to_alert": ("👨‍⚕️ تخصصات يجب إخطارها", ""),
    "equipment": ("🩺 معدات وأجهزة", ""),
    "tests_and_imaging": ("🔬 فحوصات وأشعة", ""),
    "medications_to_prepare": ("💊 أدوية يُحتمل تجهيزها", "للتجهيز فقط — القرار والجرعات للطبيب المعالج"),
    "precautions": ("⚠️ احتياطات", ""),
}
READINESS_CATALOG = {
    "immediate_actions": [
        "تأمين مجرى الهواء", "إعطاء أكسجين", "تركيب خط وريدي", "توصيل مونيتور قلب", "تخطيط قلب فوري",
        "إيقاف النزيف بالضغط المباشر", "تثبيت الرقبة والعمود الفقري", "تجهيز غرفة إنعاش", "تجهيز غرفة صدمات",
        "عزل المريض", "قياس سكر الدم", "تفعيل بروتوكول الجلطة", "تفعيل بروتوكول السكتة", "تفعيل بروتوكول الصدمات", "تجهيز نقل دم",
    ],
    "staff_to_alert": [
        "طبيب طوارئ", "تمريض طوارئ", "طبيب أطفال", "طبيب قلب", "جراح عام", "جراح عظام", "طبيب مخ وأعصاب",
        "طبيب نساء وتوليد", "طبيب تخدير وعناية", "أخصائي سموم", "بنك الدم", "قسم الأشعة", "الأمن",
    ],
    "equipment": [
        "جهاز مونيتور", "جهاز تنفس صناعي", "أنبوبة أكسجين وقناع", "جهاز تخطيط قلب", "جهاز صدمات كهربائية",
        "جهاز شفط", "جهاز سونار طوارئ", "عربة إنعاش", "مجموعة جراحية صغيرة", "جبائر وتثبيت", "جهاز قياس السكر", "مضخة سوائل",
    ],
    "tests_and_imaging": [
        "تحاليل دم أساسية", "إنزيمات القلب (تروبونين)", "غازات الدم", "سكر الدم", "وظائف الكلى", "تحليل بول",
        "أشعة سينية", "أشعة مقطعية", "سونار", "اختبار حمل", "فصيلة الدم والتصالب",
    ],
    "medications_to_prepare": [
        "أكسجين", "محلول وريدي", "أسبرين", "مسكن وريدي", "مضاد قيء", "موسع شعب هوائية", "أدرينالين",
        "مضاد حساسية", "مضاد تشنج", "خافض حرارة", "مضاد حيوي وريدي", "جلوكوز", "أنسولين", "مضاد تجلط",
    ],
    "precautions": [
        "عزل تنفسي", "حساسية محتملة", "حالة حمل", "طفل", "كبير سن", "مريض سكر", "مريض قلب",
        "خطر سقوط", "احتمال إصابة بالعمود الفقري", "مراقبة الوعي باستمرار", "مراقبة التنفس باستمرار", "خطر نزيف", "خطر تشنج",
    ],
}

def _norm_txt(s):
    return re.sub(r"[\s\-–—_]+", " ", str(s or "")).strip()

def match_department(raw, tier):
    n = _norm_txt(raw)
    for d in DEPT_CATALOG:
        if _norm_txt(d) == n:
            return d, False
    for d in DEPT_CATALOG:
        if n and _norm_txt(d) in n:
            return d, False
    return DEPT_DEFAULT_BY_TIER[tier], True

def clean_checklist(key, items):
    allowed = {_norm_txt(x): x for x in READINESS_CATALOG[key]}
    out, dropped = [], 0
    for it in (items or []):
        k = _norm_txt(it)
        if k in allowed and allowed[k] not in out:
            out.append(allowed[k])
        else:
            dropped += 1
    return out, dropped

def _catalog_prompt():
    lines = ["", "القوائم المعتمدة (انسخ منها حرفياً ولا تخترع عناصر جديدة):",
             "الأقسام (target_department): " + " | ".join(DEPT_CATALOG)]
    for key, _ in PREP_LABELS:
        lines.append(f"{key}: " + " | ".join(READINESS_CATALOG[key]))
    return "\n".join(lines) + "\n"

TIER_MIN_SCORE = {"RED": 9, "ORANGE": 5, "GREEN": 1}
TIER_RANK = {"GREEN": 0, "ORANGE": 1, "RED": 2}

def normalize_decision(d: TriageDecision):
    score = max(1, min(10, int(d.risk_score)))
    tier_from_score = "RED" if score >= 9 else ("ORANGE" if score >= 5 else "GREEN")
    tier = tier_from_score
    if TIER_RANK[d.tier] > TIER_RANK[tier_from_score]:
        tier = d.tier
        score = max(score, TIER_MIN_SCORE[tier])
    return tier, score

def _parse_int(v, lo, hi):
    try:
        n = int(str(v).strip())
    except Exception:
        return None
    return n if lo <= n <= hi else None

def _parse_float(v, lo, hi):
    try:
        n = float(str(v).strip().replace(',', '.'))
    except Exception:
        return None
    return round(n, 1) if lo <= n <= hi else None

def calculate_map(sbp, dbp):
    if sbp is None or dbp is None or not (40 <= sbp <= 300 and 20 <= dbp <= 200) or sbp <= dbp or (sbp - dbp) < 5:
        return None, None, "NA"
    exact = dbp + (sbp - dbp) / 3.0
    rounded = int(round(exact))
    st = "RED" if rounded < 60 else ("ORANGE" if (rounded < 65 or rounded >= 135) else ("WATCH" if (rounded < 70 or rounded >= 110) else "OK"))
    return rounded, round(exact, 2), st

def _pediatric_thresholds(age):
    if age == 0:
        return (30, 53, 60, 70, 20,  100, 160, 180, 200, 80,  70, 75, 1.4, 1.8)
    elif 1 <= age <= 2:
        return (22, 37, 45, 55, 15,  90, 150, 165, 185, 75,   70 + 2 * age, 70 + 2 * age + 8, 1.3, 1.6)
    elif 3 <= age <= 5:
        return (20, 28, 35, 45, 12,  80, 140, 155, 175, 65,   70 + 2 * age, 70 + 2 * age + 8, 1.2, 1.5)
    elif 6 <= age <= 12:
        return (18, 25, 30, 38, 10,  70, 120, 135, 155, 55,   70 + 2 * age, 70 + 2 * age + 10, 1.0, 1.3)
    else:
        return (12, 20, 25, 30, 8,   60, 90, 111, 131, 40,    90, 100, 0.9, 1.2)

_SEV = {"NA": 0, "OK": 1, "WATCH": 2, "ORANGE": 3, "RED": 4}

def assess_vitals(age, sbp, dbp, spo2, rr=None, temp=None, hr=None, avpu=None, gcs=None):
    floors, reasons = [], []
    b = c = d_st = "NA"
    is_peds = (age is not None and age < 13)
    (rr_lo, rr_hi, rr_or, rr_rd, rr_rd_lo, hr_lo, hr_hi, hr_or, hr_rd, hr_rd_lo, sbp_hypo, sbp_bord, si_or, si_rd) = _pediatric_thresholds(age if age is not None else 30)

    if spo2 is not None:
        if spo2 < 85:
            b = "RED"; floors.append(("RED", 10)); reasons.append(f"🫁 SpO₂ {spo2}% — نقص أكسجة شديد جداً (<85%)")
        elif spo2 < 90:
            b = "RED"; floors.append(("RED", 9)); reasons.append(f"🫁 SpO₂ {spo2}% — نقص أكسجة حاد (<90%)")
        elif spo2 < 94:
            b = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"🫁 SpO₂ {spo2}% — نقص أكسجة (90–93%)")
        elif spo2 < 96:
            b = "WATCH"; reasons.append(f"🫁 SpO₂ {spo2}% — حدّي (94–95%)")
        else:
            b = "OK"

    if rr is not None:
        age_lbl = f" (معيار عمر {age} سنة)" if is_peds else ""
        if rr <= rr_rd_lo:
            rs = "RED"; floors.append(("RED", 9)); reasons.append(f"🫁 معدل التنفس {rr}/د — بطيء جداً / فشل تنفسي وشيك{age_lbl}")
        elif rr >= rr_rd:
            rs = "RED"; floors.append(("RED", 9)); reasons.append(f"🫁 معدل التنفس {rr}/د — تسارع تنفسي خطير{age_lbl}")
        elif rr >= rr_or:
            rs = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"🫁 معدل التنفس {rr}/د — تسارع تنفسي يحتاج تقييماً عاجلاً{age_lbl}")
        elif rr > rr_hi or rr < rr_lo:
            rs = "WATCH"; reasons.append(f"🫁 معدل التنفس {rr}/د — خارج المدى الطبيعي للعمر{age_lbl}")
        else:
            rs = "OK"
        b = max((b, rs), key=_SEV.get)

    e = "NA"
    if temp is not None:
        if temp < 32.0:
            e = "RED"; floors.append(("RED", 9)); reasons.append(f"🌡️ الحرارة {temp:g}°C — انخفاض شديد (<32)")
        elif temp <= 35.0:
            e = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"🌡️ الحرارة {temp:g}°C — انخفاض حرارة (≤35)")
        elif temp >= 41.0:
            e = "RED"; floors.append(("RED", 9)); reasons.append(f"🌡️ الحرارة {temp:g}°C — ارتفاع شديد جداً (≥41)")
        elif temp >= 40.0:
            e = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"🌡️ الحرارة {temp:g}°C — ارتفاع شديد (≥40)")
        elif age is not None and age < 1 and temp >= 38.0:
            e = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"🌡️ الحرارة {temp:g}°C — حمى عند رضيع (<1 سنة)")
        elif temp >= 38.0:
            e = "WATCH"; reasons.append(f"🌡️ الحرارة {temp:g}°C — حمى")
        else:
            e = "OK"

    if sbp is not None:
        age_lbl = f" بالنسبة لعمر {age} سنة (حد الهبوط {sbp_hypo})" if is_peds else ""
        if sbp < sbp_hypo - 15:
            c = "RED"; floors.append(("RED", 10)); reasons.append(f"🩸 الضغط الانقباضي {sbp} — صدمة لا تعويضية شديدة{age_lbl}")
        elif sbp < sbp_hypo:
            c = "RED"; floors.append(("RED", 9)); reasons.append(f"🩸 الضغط الانقباضي {sbp} — هبوط ضغط (صدمة){age_lbl}")
        elif sbp < sbp_bord:
            c = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"🩸 الضغط الانقباضي {sbp} — انخفاض حدّي{age_lbl}")
        else:
            c = "OK"

    if not is_peds and ((sbp is not None and sbp >= 180) or (dbp is not None and dbp >= 120)):
        if c in ("NA", "OK", "WATCH"):
            c = "ORANGE"
        floors.append(("ORANGE", 7))
        reasons.append(f"🩸 ارتفاع شديد في الضغط ({sbp if sbp is not None else '؟'}/{dbp if dbp is not None else '؟'}) — أزمة ضغط محتملة")
    if c == "NA" and dbp is not None:
        c = "OK"

    hr_st = "NA"
    if hr is not None:
        age_lbl = f" (معيار عمر {age} سنة)" if is_peds else ""
        if hr <= hr_rd_lo:
            hr_st = "RED"; floors.append(("RED", 9)); reasons.append(f"💓 النبض {hr}/د — بطء قلب خطير{age_lbl}")
        elif hr >= hr_rd:
            hr_st = "RED"; floors.append(("RED", 9)); reasons.append(f"💓 النبض {hr}/د — تسارع قلب شديد خطير{age_lbl}")
        elif hr >= hr_or or hr < hr_lo:
            hr_st = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append(f"💓 النبض {hr}/د — خارج المدى التشغيلي للعمر{age_lbl}")
        elif hr > hr_hi:
            hr_st = "WATCH"; reasons.append(f"💓 النبض {hr}/د — تسارع حدّي{age_lbl}")
        else:
            hr_st = "OK"
        c = max((c, hr_st), key=_SEV.get)

    si = None
    if hr is not None and sbp is not None and sbp > 0:
        si = round(hr / sbp, 2)
        if si >= si_rd:
            c = max((c, "RED"), key=_SEV.get); floors.append(("RED", 9))
            reasons.append(f"⚠️ مؤشر الصدمة SI={si} (النبض {hr} / الضغط {sbp}) — اشتباه صدمة/نزيف")
        elif si > si_or:
            c = max((c, "ORANGE"), key=_SEV.get); floors.append(("ORANGE", 8))
            reasons.append(f"⚠️ مؤشر الصدمة SI={si} (>{si_or}) — احتمال صدمة مبكرة")

    if avpu:
        avpu_u = str(avpu).upper().strip()
        if avpu_u == "U":
            d_st = "RED"; floors.append(("RED", 10)); reasons.append("🧠 مستوى الوعي AVPU = U (لا يستجيب / غيبوبة) — حالة حرجة فوراً")
        elif avpu_u == "P":
            d_st = "RED"; floors.append(("RED", 9)); reasons.append("🧠 مستوى الوعي AVPU = P (يستجيب للألم فقط) — تدهور شديد في الوعي")
        elif avpu_u == "V":
            d_st = "ORANGE"; floors.append(("ORANGE", 7)); reasons.append("🧠 مستوى الوعي AVPU = V (يستجيب للصوت فقط) — اضطراب وعي عاجل")
        elif avpu_u == "A":
            d_st = "OK"

    if gcs is not None:
        if gcs <= 8:
            d_st = max((d_st, "RED"), key=_SEV.get); floors.append(("RED", 10))
            reasons.append(f"🧠 مقياس غلاسكو GCS={gcs}/15 (≤ 8) — غيبوبة / خطر فقدان مجرى الهواء")
        elif gcs <= 13:
            d_st = max((d_st, "ORANGE"), key=_SEV.get); floors.append(("ORANGE", 7))
            reasons.append(f"🧠 مقياس غلاسكو GCS={gcs}/15 (9–13) — اضطراب متوسط في الوعي")
        elif gcs == 14:
            d_st = max((d_st, "WATCH"), key=_SEV.get); reasons.append("🧠 مقياس غلاسكو GCS=14/15 — تشوش طفيف")
        else:
            d_st = max((d_st, "OK"), key=_SEV.get)

    map_val, map_exact, map_status = calculate_map(sbp, dbp)
    if map_val is not None and not is_peds:
        if map_val < 60:
            reasons.append(f"🩸 متوسط الضغط الشرياني التقديري MAP={map_val} mmHg — منخفض بشدة؛ يشير لضعف التروية")
        elif map_val < 65:
            reasons.append(f"🩸 متوسط الضغط الشرياني التقديري MAP={map_val} mmHg — منخفض؛ قيّم التروية")
        elif map_val >= 135:
            reasons.append(f"🩸 متوسط الضغط الشرياني التقديري MAP={map_val} mmHg — مرتفع بشدة")

    floor, min_score = None, 0
    for tier, ms in floors:
        if floor is None or TIER_RANK[tier] > TIER_RANK[floor] or (tier == floor and ms > min_score):
            floor, min_score = tier, ms
    return {"floor": floor, "min_score": min_score, "reasons": reasons, "b": b, "c": c, "d": d_st, "e": e,
            "spo2": spo2, "sbp": sbp, "dbp": dbp, "rr": rr, "temp": temp, "hr": hr, "si": si,
            "avpu": avpu, "gcs": gcs, "map": map_val, "map_exact": map_exact, "map_status": map_status}

def apply_vitals_floor(tier, score, vit):
    f = vit["floor"]
    if f and TIER_RANK[f] > TIER_RANK[tier]:
        return f, max(score, vit["min_score"])
    if f and TIER_RANK[f] == TIER_RANK[tier]:
        return tier, max(score, vit["min_score"])
    return tier, score

class _RulesOnlyDecision:
    def __init__(self, tier, vit):
        txt = "؛ ".join(r.split(" ", 1)[1] if " " in r else r for r in vit["reasons"])
        self.clinical_summary = "تقييم مبدئي بالعلامات الحيوية والوعي فقط: " + txt
        self.target_department = DEPT_DEFAULT_BY_TIER[tier]
        self.other_note = ""
        for k, _ in PREP_LABELS:
            setattr(self, k, [])

TRIAGE_INSTRUCTION = """
أنت مساعد ضمن منظومة RELAY يقترح **مسودة** فرز مبدئي مستندة إلى معايير ESI (Emergency Severity Index) وبروتوكولات الأطفال PALS (Pediatric Advanced Life Support)، وتُراجَع وتُعتمد بشرياً.
افهم اللهجة المصرية والعامية الطبية. قيّم الحالة بحيادية ودقة طبية دون أي أرقام افتراضية:
1. RED (9-10): مهددة للحياة فوراً (بتر، نزيف حاد، توقف تنفس، غيبوبة GCS<=8 أو AVPU=U/P، ألم صدر عاصر، صدمة).
2. ORANGE (5-8): عاجلة وعالية الخطورة (ألم بطن حاد، مغص كلوي حاد، كسر غير مستقر، حمى شديدة، اضطراب وعي AVPU=V).
3. GREEN (1-4): رعاية روتينية أو مستقرة.
4. راعِ عمر المريض ونوعه (Gender: Male/Female): إذا كان طفلاً (<13 سنة) اختر "طوارئ الأطفال" ما لم تكن صدمات كبرى/حروق/سموم؛ وإذا كانت أنثى في سن الإنجاب بألم بطن حاد أو نزيف مهبلي فكّر في اختبار الحمل وطوارئ النساء والتوليد.
5. اختر العناصر حرفياً من القوائم المعتمدة أدناه فقط. medications_to_prepare للتجهيز فقط دون جرعات.
6. الأدوية الحالية للمريض (بما فيها الفئات الدوائية غير محددة الاسم مثل أدوية الصرع أو السيولة) سياق مهم: اعكس مخاطرها (مثل خطر النزيف أو التشنج) في الاحتياطات المناسبة.
""" + _catalog_prompt()

def _triage_config(model, thinking=True, language="ar"):
    lang_inst = (
        "\n\nLANGUAGE REQUIREMENT: Write clinical_summary and other_note in clear, concise English. Keep target_department and checklist items in Arabic exactly as listed."
        if str(language).lower().startswith("en") else
        "\n\nمتطلب اللغة: اكتب clinical_summary و other_note بالعربية الواضحة الموجزة، مع إبقاء القسم وعناصر القوائم مطابقة حرفياً للقوائم العربية المعتمدة."
    )
    kwargs = dict(system_instruction=TRIAGE_INSTRUCTION + lang_inst, response_mime_type="application/json", response_schema=TriageDecision, temperature=0.0, max_output_tokens=900)
    th = _thinking_cfg(model) if thinking else None
    if th:
        kwargs["thinking_config"] = th
    return types.GenerateContentConfig(**kwargs)

def _parse_triage(resp):
    return TriageDecision.model_validate_json(resp.text)

# ============================================================
# 4b. الاستخراج الذكي السريع للصوت والنص (ثنائي اللغة + النوع + الفئات الدوائية)
# ============================================================
class ExtractedCaseFields(BaseModel):
    patient_name: Optional[str] = Field(default=None, description="الاسم الكامل للمريض إن ذُكر، بدون كلمات مثل 'اسمه' أو 'المريض'. Null إذا لم يُذكر.")
    age: Optional[int] = Field(default=None, description="عمر المريض بالسنوات (0-130). إذا كان رضيعاً بالشهور ضع 0. Null إذا لم يُذكر.")
    gender: Optional[Literal["Male", "Female"]] = Field(default=None, description="نوع المريض: Male (ذكر/رجل/شاب/ولد) أو Female (أنثى/سيدة/بنت/حامل). Null إذا لم يُذكر.")
    national_id: Optional[str] = Field(default=None, description="الرقم القومي المصري المكون من 14 رقماً متصلاً (يبدأ بـ 2 أو 3). Null إذا لم يكتمل 14 رقماً.")
    sbp: Optional[int] = Field(default=None, description="الضغط الانقباضي Systolic BP (40-300).")
    dbp: Optional[int] = Field(default=None, description="الضغط الانبساطي Diastolic BP (20-200).")
    hr: Optional[int] = Field(default=None, description="معدل النبض Heart Rate (20-260).")
    spo2: Optional[int] = Field(default=None, description="تشبع الأكسجين SpO2 (50-100).")
    rr: Optional[int] = Field(default=None, description="معدل التنفس Respiratory Rate (4-80).")
    temp: Optional[float] = Field(default=None, description="درجة الحرارة (28-45 مئوية أو 80-115 فهرنهايت).")
    avpu: Optional[Literal["A", "V", "P", "U"]] = Field(default=None, description="مستوى الوعي AVPU: A, V, P, أو U.")
    gcs: Optional[int] = Field(default=None, description="درجة مقياس غلاسكو للغيبوبة GCS (3-15).")
    medications: Optional[str] = Field(default=None, description="أسماء الأدوية وأي فئات دوائية غير معروفة الاسم مفصولة بفاصلة، دون سحب أي أعراض أو جمل تالية.")
    eta: Optional[int] = Field(default=None, description="وقت الوصول المتوقع بالدقائق (1-180).")

EXTRACT_SYSTEM_PROMPT = """
أنت محلل بيانات طبية فائق السرعة والدقة لسيارات الإسعاف ضمن نظام RELAY (تعمل بالعربية العامية المصرية والإنجليزية).
استخرج البيانات التالية بدقة صارمة:

1. **الاسم (patient_name):** التقط اسم الشخص حتى لو ذُكر مباشرة في بداية الكلام دون كلمة "اسمه" (مثال: "محمد علي محمود 35 سنة..." -> "محمد علي محمود"). استبعد أي صفات أو أفعال.
2. **العمر (age) والنوع (gender):**
   - العمر: رقم صحيح بالسنوات ("23 سنة" أو "عنده 23" -> 23).
   - النوع: حدد `Male` إذا ذُكر (مريض، راجل، شاب، ولد، ذكر، Male، man، boy) أو من اسم مذكر واضح؛ وحدد `Female` إذا ذُكر (مريضة، ست، سيدة، بنت، أنثى، حامل، Female، woman، girl) أو من اسم مؤنث واضح.
3. **الرقم القومي (national_id):** 14 رقماً متصلاً يبدأ بـ 2 أو 3. اجمعه إذا نُطق مقطعاً أو بالحروف.
4. **الأدوية والفئات الدوائية (medications) — قاعدة العزل والشمول الطبي:**
   - استخرج **أسماء الأدوية وأي فئة علاجية مذكورة** وتوقف فوراً عند انتهاء ذكر الأدوية، ولا تضمّن أي أعراض أو علامات حيوية تُقال بعدها.
   - حوّل أسماء الأدوية المنطوقة بالعربي إلى اسمها الإنجليزي الدقيق (مثل: بنادول -> Panadol (Paracetamol)، كونكور -> Concor (Bisoprolol)، بلافكس -> Plavix (Clopidogrel)، ماريفان -> Marevan (Warfarin)، زاريلتو -> Xarelto (Rivaroxaban)، إلكويس -> Eliquis (Apixaban)، أسبرين/أسبوسيد -> Aspirin، كليكسان -> Clexane (Enoxaparin)، جلوكوفاج -> Glucophage (Metformin)، أنسولين -> Insulin، لازيكس -> Lasix (Furosemide)، كابوتن -> Capoten (Captopril)، فينتولين -> Ventolin (Salbutamol)، أوجمنتين -> Augmentin).
   - **مهم جداً للتداخلات الدوائية (Drug Interactions):** إذا ذكر المتحدث فئة دوائية بدون اسم محدد (مثل: *"أدوية للصرع مش معروفة"*، *"علاج للسيولة"*، *"أدوية قلب"*، *"أدوية ضغط"*، *"أدوية سكر"*، *"أدوية نفسية"*، *"unknown antiepileptic drugs"*)، **اكتبها بوضوح في خانة الأدوية** ولا تحذفها أبداً! (مثال: إذا قال *"بياخد بنادول وأدوية للصرع مش معروفة"* -> اكتب: `"Panadol (Paracetamol), أدوية صرع غير معروفة (Unknown Antiepileptics)"`).
5. **العلامات الحيوية والوعي:**
   - الضغط: فقط الأرقام المنطقية للضغط (مثل 120 على 80 أو 90/60).
   - الوعي AVPU: فايق/واعي/Alert -> A | يستجيب للصوت/مدروخ/Voice -> V | يستجيب للألم/Pain -> P | فاقد الوعي/غيبوبة/Unresponsive -> U.
"""

def _extract_config(model, thinking=True):
    kwargs = dict(system_instruction=EXTRACT_SYSTEM_PROMPT, response_mime_type="application/json", response_schema=ExtractedCaseFields, temperature=0.0, max_output_tokens=450)
    th = _thinking_cfg(model) if thinking else None
    if th:
        kwargs["thinking_config"] = th
    return types.GenerateContentConfig(**kwargs)

def _parse_extracted(resp):
    return ExtractedCaseFields.model_validate_json(resp.text)

def extract_fields_v1(raw_text, ui_lang="ar"):
    try:
        txt = str(raw_text or "").strip()
        if len(txt) < 2:
            return JSON({"ok": False, "error": "النص قصير جداً"})
        extracted, model_used = generate_with_fallback(
            contents=f"UI_LANG={ui_lang}\nTEXT:\n{txt}",
            make_config=_extract_config,
            parse=_parse_extracted,
            budget_sec=12,
            rounds=2,
        )
        data = extracted.model_dump()
        if data.get("national_id"):
            digits = re.sub(r"\D", "", str(data["national_id"]).translate(_AR_DIGITS))
            data["national_id"] = digits if len(digits) == 14 else None
        return JSON({"ok": True, "fields": data, "model": model_used})
    except Exception as e:
        return JSON({"ok": False, "error": str(e)})

# ============================================================
# 4c. تنبيهات تليجرام والمصادقة (محصّنة ضد 409 Conflict والمهلة)
# ============================================================
NOTIFY_TIERS = {"RED", "ORANGE"}
_SECRET_REASON = {}

def _secret(name):
    try:
        v = (userdata.get(name) or "").strip().strip('"').strip("'").strip()
        return v or None
    except Exception as e:
        kind = type(e).__name__ + " " + str(e)
        _SECRET_REASON[name] = f"فعّل زر Notebook access بجوار {name} في Secrets" if "Access" in kind else f"لا يوجد سر باسم {name} في Secrets"
        return None

TG_TOKEN = _secret("TELEGRAM_BOT_TOKEN")
if TG_TOKEN and TG_TOKEN.lower().startswith("bot") and ":" in TG_TOKEN:
    TG_TOKEN = TG_TOKEN[3:]
TG_CHAT_ID = _secret("TELEGRAM_CHAT_ID")

def _tg_off_reason():
    return _SECRET_REASON.get("TELEGRAM_BOT_TOKEN", "لم يُضبط TELEGRAM_BOT_TOKEN")

def _tg_api(method, payload=None, timeout=8):
    r = requests.post(f"https://api.telegram.org/bot{TG_TOKEN}/{method}", json=payload or {}, timeout=timeout)
    return r.json()

def _discover_chat_id():
    global TG_CHAT_ID
    if TG_CHAT_ID:
        return TG_CHAT_ID
    _tg_api("deleteWebhook", {"drop_pending_updates": False}, timeout=6)
    data = _tg_api("getUpdates", timeout=8)
    for upd in reversed(data.get("result", [])):
        chat = (upd.get("message") or upd.get("my_chat_member") or {}).get("chat") or {}
        if chat.get("id"):
            TG_CHAT_ID = str(chat["id"])
            return TG_CHAT_ID
    return None

TIER_EMOJI = {"RED": "🔴", "ORANGE": "🟠", "GREEN": "🟢"}
TIER_AR = {"RED": "حالة حمراء - إنعاش فوري", "ORANGE": "حالة برتقالية - عاجلة", "GREEN": "حالة خضراء - روتينية"}
PREP_EMOJI = {
    "immediate_actions": "🚑 إجراءات فورية", "staff_to_alert": "👨‍⚕️ إخطار", "equipment": "🩺 معدات",
    "tests_and_imaging": "🔬 فحوصات", "medications_to_prepare": "💊 أدوية (للتجهيز فقط)", "precautions": "⚠️ احتياطات",
}

def _case_number():
    try:
        with open(CSV_FILE, mode='r', encoding='utf-8-sig') as f:
            return max(0, sum(1 for _ in csv.reader(f)) - 1)
    except Exception:
        return 0

def build_telegram_message(case_no, patient_id, tier, score, age, gender, time_str, summary, dept, prep, vit=None,
                           other_note="", model_used="", reviewer="", edited=False, meds="", eta=None):
    e = html.escape
    eta_txt = f" | ⏱️ الوصول خلال: <b>{eta} دقيقة</b>" if eta is not None else ""
    g_txt = f" | النوع: {e(gender)}" if gender and gender != "-" else ""
    lines = [
        "🚑 <b>RELAY — تنبيه مسبق من المسعف (رُوجع بشرياً)</b>",
        f"{TIER_EMOJI[tier]} <b>{TIER_AR[tier]}</b> — #{case_no} | 🆔 <code>{e(patient_id)}</code>",
        f"🕒 {e(time_str)} | العمر: {e(str(age))}{g_txt} | الخطورة: <b>{score}/10</b>{eta_txt}",
    ]
    if vit:
        vparts = []
        if vit.get("sbp") is not None or vit.get("dbp") is not None:
            vparts.append(f"الضغط {vit.get('sbp') if vit.get('sbp') is not None else '؟'}/{vit.get('dbp') if vit.get('dbp') is not None else '؟'}")
        if vit.get("map") is not None:
            vparts.append(f"MAP {vit['map']} mmHg")
        if vit.get("hr") is not None:
            vparts.append(f"النبض {vit['hr']}/د")
        if vit.get("si") is not None:
            vparts.append(f"SI {vit['si']}")
        if vit.get("spo2") is not None:
            vparts.append(f"SpO₂ {vit['spo2']}%")
        if vit.get("rr") is not None:
            vparts.append(f"RR {vit['rr']}")
        if vit.get("avpu"):
            vparts.append(f"AVPU {vit['avpu']}")
        if vit.get("gcs") is not None:
            vparts.append(f"GCS {vit['gcs']}")
        if vit.get("temp") is not None:
            vparts.append(f"{vit['temp']:g}°C")
        if vparts:
            lines.append("🩺 " + " | ".join(vparts))
        lines += [f"⚠️ {e(r)}" for r in vit.get("reasons", []) if "حدّي" not in r or vit.get("floor")]
    lines += [f"📝 {e(summary)}", f"🏥 <b>القسم:</b> {e(dept)}"]
    for key, title in PREP_EMOJI.items():
        items = prep.get(key) or []
        if items:
            lines += ["", f"<b>{title}</b>"] + [f"• {e(i)}" for i in items]
    if meds:
        lines += ["", f"💊 الأدوية الحالية: {e(meds)}"]
    if other_note:
        lines += ["", f"📌 ملاحظة: {e(other_note)}"]
    lines += ["", f"🤖 النموذج: {e(model_used)}" + (" | ✎ عُدّل بشرياً" if edited else ""), f"✅ المراجع: {e(reviewer)}"]
    return "\n".join(lines)

def notify_telegram(tier, text):
    if not TG_TOKEN:
        return f"التنبيهات متوقفة: {_tg_off_reason()}"
    if tier not in NOTIFY_TIERS:
        return "لا تنبيه لهذه الفئة (تُرسل تنبيهات RED و ORANGE فقط)"
    try:
        chat = _discover_chat_id()
        if not chat:
            return "افتح البوت في تليجرام وأرسل له /start أو أضف TELEGRAM_CHAT_ID في Secrets"
        res = _tg_api("sendMessage", {"chat_id": chat, "text": text, "parse_mode": "HTML"}, timeout=8)
        if res.get("ok"):
            return "تم إرسال التنبيه إلى تليجرام ✅"
        return f"فشل الإرسال من تليجرام: {res.get('description', 'خطأ غير معروف')}"
    except Exception as ex:
        return f"تعذر الاتصال بتليجرام: {ex}"

TG_STATE = "⏳ جاري فحص تليجرام..." if TG_TOKEN else f"غير مفعّل: {_tg_off_reason()}"

def _telegram_startup_worker():
    global TG_STATE
    if not TG_TOKEN:
        return
    try:
        me = _tg_api("getMe", timeout=8)
        if not me.get("ok"):
            TG_STATE = f"التوكن مرفوض: {me.get('description', 'تحقق من التوكن')}"
            return
        bot_user = me["result"]["username"]
        chat = _discover_chat_id()
        if not chat:
            TG_STATE = f"البوت @{bot_user} جاهز — افتحه وأرسل /start أو أضف TELEGRAM_CHAT_ID"
            return
        res = _tg_api("sendMessage", {"chat_id": chat, "text": "✅ RELAY — Bridging the Gap Before Arrival متصل بنجاح."}, timeout=8)
        TG_STATE = f"متصل ✅ (@{bot_user})" if res.get("ok") else f"فشل الإرسال: {res.get('description')}"
    except Exception as ex:
        TG_STATE = f"تعذر الاتصال: {ex}"

def notify_telegram_bounded(tier, text, wait_sec=8):
    box = {}
    def work():
        try:
            box["r"] = notify_telegram(tier, text)
        except Exception as ex:
            box["r"] = f"تعذر إرسال التنبيه: {ex}"
    t = threading.Thread(target=work, daemon=True)
    t.start()
    t.join(wait_sec)
    return box.get("r", "انتهت مهلة الانتظار (8ث) أثناء الإرسال لتليجرام")

def authenticate_paramedic_v1(code, pin):
    code = str(code or "").strip().upper()[:32]
    pin = str(pin or "").strip()[:32]
    account = PARAMEDIC_ACCOUNTS.get(code)
    if not account or not _pysecrets.compare_digest(str(account.get("pin", "")), pin):
        return JSON({"ok": False, "error": "كود المسعف أو الرقم السري غير صحيح."})
    return JSON({"ok": True, "code": code, "name": str(account.get("name") or code)[:60]})

def emergency_alert_v1(paramedic=""):
    msg = (
        "🚨 <b>RELAY — EMERGENCY ALERT (RED FLAG)</b>\n"
        "🟥 <b>CODE BLUE READINESS REQUESTED</b>\n"
        f"👨‍🚑 Paramedic: {html.escape(str(paramedic or 'Unidentified')[:60])}\n"
        "⚠️ This is an app notification only. Activate the facility's official emergency/code-blue pathway separately."
    )
    result = notify_telegram_bounded("RED", msg, wait_sec=8)
    return JSON({"ok": result.startswith("تم إرسال"), "message": result})

def status_v1():
    return JSON({"ok": True, "models": MODEL_CHAIN[:3], "telegram": TG_STATE, "version": APP_PROMPT_VERSION, "locked": TRIAGE_LOCKED_MODEL})

_AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

def _redact_ids(text):
    t = str(text or "").translate(_AR_DIGITS)
    t = re.sub(r"(?<!\d)[23](?:\s?\d){13}(?!\d)", "[رقم قومي محجوب]", t)
    t = re.sub(r"(?<!\d)01[0125](?:\s?\d){8}(?!\d)", "[هاتف محجوب]", t)
    return t

def _redact_for_model(text, name):
    t = _redact_ids(text)
    n = str(name or "").strip()
    if n and n not in ("غير مسجل", "-"):
        for tok in n.split():
            if len(tok) >= 2:
                t = re.sub(r"(?<![\u0621-\u064A])" + re.escape(tok) + r"(?![\u0621-\u064A])", "[اسم محجوب]", t)
    t = re.sub(r"(?:اسمه|اسمها|اسم المريض|patient name)\s*[:=]?\s*[\u0621-\u064AA-Za-z]+(?:\s+[\u0621-\u064AA-Za-z]+){0,2}", "[اسم محجوب]", t, flags=re.I)
    return t

DRAFTS = {}

def _prune_drafts():
    now = time.time()
    for k in [k for k, v in DRAFTS.items() if now - v["ts"] > DRAFT_TTL_SEC]:
        DRAFTS.pop(k, None)
    while len(DRAFTS) > 200:
        DRAFTS.pop(next(iter(DRAFTS)))

def _triage_impl(p_name, p_age, p_national_id, user_text, p_sbp="", p_dbp="", p_spo2="", p_rr="", p_temp="", p_meds="", p_language="ar", p_hr="", p_eta="", p_gender=""):
    user_text = _redact_ids(user_text)
    age_n = _parse_int(p_age, 0, 130)
    gender_disp = _format_gender_ar(p_gender)
    sbp = _parse_int(p_sbp, 40, 300)
    dbp = _parse_int(p_dbp, 20, 200)
    spo2 = _parse_int(p_spo2, 50, 100)
    rr = _parse_int(p_rr, 4, 80)
    temp = _parse_float(p_temp, 28.0, 45.0)
    hr = _parse_int(p_hr, 20, 260)
    eta = _parse_int(p_eta, 1, 180)
    meds = _redact_ids(str(p_meds or "").strip())[:300]

    consciousness_match = re.search(r"\[Consciousness:\s*([^\]]+)\]", user_text, re.I)
    consciousness_text = consciousness_match.group(1) if consciousness_match else ""
    avpu_match = re.search(r"\bAVPU=([AVPU])\b", consciousness_text, re.I)
    gcs_match = re.search(r"\bGCS=(1[0-5]|[3-9])\b", consciousness_text, re.I)
    avpu_value = avpu_match.group(1).upper() if avpu_match else None
    gcs_value = int(gcs_match.group(1)) if gcs_match else None

    vit = assess_vitals(age_n, sbp, dbp, spo2, rr, temp, hr, avpu=avpu_value, gcs=gcs_value)

    vit_parts = []
    if sbp is not None and dbp is not None and vit.get("map") is not None:
        vit_parts.append(f"ضغط الدم: {sbp}/{dbp} ملم زئبق | MAP: {vit['map']} ملم زئبق")
    elif sbp is not None or dbp is not None:
        vit_parts.append(f"ضغط الدم غير مكتمل: {sbp if sbp is not None else 'مفقود'}/{dbp if dbp is not None else 'مفقود'} ملم زئبق")
    else:
        vit_parts.append("ضغط الدم و MAP: غير مقاسين")

    if hr is not None:
        vit_parts.append(f"النبض HR: {hr}/دقيقة" + (f" (SI={vit['si']})" if vit.get("si") is not None else ""))
    if spo2 is not None:
        vit_parts.append(f"تشبع الأكسجين SpO2: {spo2}%")
    if rr is not None:
        vit_parts.append(f"معدل التنفس: {rr}/دقيقة")
    if temp is not None:
        vit_parts.append(f"الحرارة: {temp:g}°C")
    if eta is not None:
        vit_parts.append(f"وقت الوصول ETA: {eta} دقيقة")
    if avpu_value:
        vit_parts.append(f"مستوى الوعي AVPU: {avpu_value}")
    if gcs_value is not None:
        vit_parts.append(f"مقياس غلاسكو GCS: {gcs_value}/15")
    vit_text = " | ".join(vit_parts) or "لم تُقَس"

    ai_ok = True
    try:
        decision, model_used = generate_with_fallback(
            contents=f"المريض: (محجوب)، العمر: {p_age}، النوع: {gender_disp}. العلامات الحيوية والوعي: {vit_text}. الأدوية الحالية: {meds or 'غير مذكورة'}. نص الشكوى: {_redact_for_model(user_text, p_name)}",
            make_config=lambda model, thinking: _triage_config(model, thinking, p_language),
            parse=_parse_triage,
            models=[TRIAGE_LOCKED_MODEL] if TRIAGE_LOCKED_MODEL else None,
        )
        tier, score = normalize_decision(decision)
    except Exception as e:
        if not vit["floor"]:
            return JSON({"error": str(e)})
        ai_ok = False
        decision = _RulesOnlyDecision(vit["floor"], vit)
        model_used = "قواعد العلامات الحيوية والوعي فقط (بدون ذكاء اصطناعي)"
        tier, score = vit["floor"], vit["min_score"]

    ai_raw = f"{tier}/{score}" if ai_ok else "بدون ذكاء اصطناعي"
    tier_before = tier
    tier, score = apply_vitals_floor(tier, score, vit)
    vitals_raised = TIER_RANK[tier] > TIER_RANK[tier_before]

    dept, dept_flag = match_department(decision.target_department, tier)
    prep, dropped = {}, 0
    for k, _ in PREP_LABELS:
        prep[k], d = clean_checklist(k, getattr(decision, k))
        dropped += d
    other_note = str(getattr(decision, "other_note", "") or "").strip()[:200]
    summary = str(decision.clinical_summary or "").strip()
    primary_model = TRIAGE_LOCKED_MODEL or MODEL_CHAIN[0]
    fallback_used = (not ai_ok) or (model_used != primary_model)

    draft_id = uuid.uuid4().hex[:12]
    patient_id = _generate_patient_id()
    _prune_drafts()
    DRAFTS[draft_id] = {
        "ts": time.time(), "patient_id": patient_id, "name": p_name, "age": p_age, "gender": gender_disp, "nid": p_national_id, "complaint": user_text,
        "sbp": sbp, "dbp": dbp, "spo2": spo2, "rr": rr, "temp": temp, "hr": hr, "eta": eta, "meds": meds, "vit": vit, "tier": tier, "score": score, "summary": summary,
        "dept": dept, "prep": prep, "other_note": other_note, "model": model_used, "ai_raw": ai_raw,
        "fallback_used": fallback_used, "ai_ok": ai_ok,
    }

    return JSON({
        "status": "draft_requires_human_review", "draft_id": draft_id, "patient_id": patient_id, "gender": gender_disp, "tier": tier, "risk_score": score,
        "clinical_summary": summary, "target_department": dept, "dept_flag": dept_flag, "dropped_items": dropped,
        **prep, "other_note": other_note,
        "vitals": {k: vit[k] for k in ("spo2", "sbp", "dbp", "map", "map_exact", "map_status", "rr", "temp", "hr", "si", "avpu", "gcs", "b", "c", "d", "e", "reasons")},
        "vitals_raised": vitals_raised, "model": model_used, "primary_model": primary_model,
        "vitals_floor": vit["floor"], "fallback_used": fallback_used, "ai_ok": ai_ok,
    })

def _confirm_impl(payload_json):
    try:
        p = json.loads(payload_json)
    except Exception:
        return JSON({"error": "بيانات التأكيد غير صالحة."})
    draft = DRAFTS.get(str(p.get("draft_id", "")))
    if not draft:
        return JSON({"error": "المسودة غير موجودة أو انتهت صلاحيتها. أنشئ مسودة جديدة."})
    if time.time() - draft["ts"] > DRAFT_TTL_SEC:
        DRAFTS.pop(str(p.get("draft_id")), None)
        return JSON({"error": "انتهت صلاحية المسودة. أنشئ مسودة جديدة."})
    if p.get("confirmed") is not True:
        return JSON({"error": "يجب تأكيد إتمام المراجعة البشرية أولاً."})
    reviewer = str(p.get("reviewer", "")).strip()[:60]
    if len(reviewer) < 2:
        return JSON({"error": "اكتب اسم أو كود المراجع المسؤول عن التأكيد."})

    tier = str(p.get("tier", "")).upper().strip()
    if tier not in TIER_AR:
        return JSON({"error": "الفئة غير صالحة."})
    try:
        score = int(p.get("score"))
    except (TypeError, ValueError):
        return JSON({"error": "درجة الخطورة غير صالحة."})
    if not 1 <= score <= 10:
        return JSON({"error": "درجة الخطورة يجب أن تكون بين 1 و 10."})
    if ((tier == "RED" and score < 9) or (tier == "ORANGE" and not 5 <= score <= 8) or (tier == "GREEN" and score > 4)):
        return JSON({"error": "الفئة ودرجة الخطورة غير متسقتين (أحمر 9-10، برتقالي 5-8، أخضر 1-4). عدّل إحداهما."})

    summary = str(p.get("summary", "")).strip()
    if not summary:
        return JSON({"error": "الملخص السريري لا يمكن أن يكون فارغاً."})
    department = str(p.get("department", "")).strip()
    if department == DEPT_OTHER:
        other_dep = str(p.get("department_other", "")).strip()[:80]
        if not other_dep:
            return JSON({"error": "حدّد القسم يدوياً عند اختيار (أخرى)."})
        final_dept = f"أخرى: {other_dep}"
    elif department in DEPT_CATALOG:
        final_dept = department
    else:
        return JSON({"error": "القسم المختار ليس من القائمة المعتمدة."})

    prep_in = p.get("prep") or {}
    prep = {}
    for k, label in PREP_LABELS:
        cleaned, dropped = clean_checklist(k, prep_in.get(k, []))
        if dropped:
            return JSON({"error": f"عنصر خارج القائمة المعتمدة في قسم ({label})."})
        prep[k] = cleaned
    other_note = str(p.get("other_note", "")).strip()[:200]

    vit = draft["vit"]
    override_reason = str(p.get("override_reason", "")).strip()[:200]
    if vit["floor"] and TIER_RANK[tier] < TIER_RANK[vit["floor"]] and len(override_reason) < 5:
        return JSON({"error": "الفئة أقل من الحد الأدنى الذي تفرضه العلامات الحيوية. اكتب سبب التخفيض في خانة (سبب التعديل)."})

    changes = []
    if tier != draft["tier"] or score != draft["score"]:
        changes.append("الفئة/الدرجة")
    if summary != draft["summary"]:
        changes.append("الملخص")
    if final_dept != draft["dept"]:
        changes.append("القسم")
    if any(prep[k] != draft["prep"][k] for k, _ in PREP_LABELS):
        changes.append("التجهيزات")
    if other_note != draft["other_note"]:
        changes.append("الملاحظة")
    edited_txt = ("نعم: " + "، ".join(changes)) if changes else "لا"

    time_now = datetime.now(CAIRO_TZ).strftime("%Y-%m-%d %H:%M:%S")
    arabic_flag = {"RED": "أحمر", "ORANGE": "برتقالي", "GREEN": "أخضر"}[tier]
    parts = [final_dept] + [f"{label}: " + "، ".join(prep[k]) for k, label in PREP_LABELS if prep[k]]
    if other_note:
        parts.append(f"ملاحظة: {other_note}")
    dep_text = " | ".join(parts)
    sbp, dbp, spo2 = draft["sbp"], draft["dbp"], draft["spo2"]
    bp_cell = f"{sbp if sbp is not None else '?'}/{dbp if dbp is not None else '?'}" if (sbp is not None or dbp is not None) else "-"
    sp_cell = str(spo2) if spo2 is not None else "-"
    map_cell = str(vit.get("map")) if vit.get("map") is not None else "-"
    model_cell = draft["model"] + (" [نموذج بديل]" if draft["ai_ok"] and draft["fallback_used"] else "")

    patient_id = draft.get("patient_id") or _generate_patient_id()
    gender_cell = draft.get("gender") or "-"
    try:
        with open(CSV_FILE, mode='a', newline='', encoding='utf-8-sig') as f:
            csv.writer(f).writerow([
                time_now, _safe_cell(patient_id), _safe_cell(draft["name"]), _safe_cell(draft["age"]), _safe_cell(gender_cell), _safe_cell(_mask_id(draft["nid"])),
                _safe_cell(summary), arabic_flag, score, _safe_cell(dep_text), bp_cell, sp_cell,
                _safe_cell(model_cell), draft["ai_raw"], edited_txt, _safe_cell(reviewer),
                _safe_cell(override_reason or "-"), APP_PROMPT_VERSION,
                draft.get("rr") if draft.get("rr") is not None else "-", draft.get("temp") if draft.get("temp") is not None else "-",
                _safe_cell(draft.get("meds") or "-"),
                draft.get("hr") if draft.get("hr") is not None else "-",
                draft.get("eta") if draft.get("eta") is not None else "-",
                draft.get("vit", {}).get("avpu") or "-",
                draft.get("vit", {}).get("gcs") if draft.get("vit", {}).get("gcs") is not None else "-",
                map_cell,
            ])
    except Exception as e:
        return JSON({"error": f"تعذر حفظ السجل: {e}"})

    try:
        notify_status = notify_telegram_bounded(
            tier,
            build_telegram_message(_case_number(), patient_id, tier, score, draft["age"], gender_cell, time_now[11:], summary, final_dept,
                                   prep, vit, other_note, model_cell, reviewer, bool(changes), draft.get("meds", ""), draft.get("eta")),
        )
    except Exception as ex:
        notify_status = f"تعذر إرسال التنبيه: {ex}"

    DRAFTS.pop(str(p.get("draft_id")), None)
    return JSON({"saved": True, "patient_id": patient_id, "notify": notify_status, "edited": edited_txt, "table_html": get_saved_rows_html()})

_CONFIRM_LOCK = threading.Lock()

def confirm_and_save_v1(payload_json):
    try:
        with _CONFIRM_LOCK:
            return _confirm_impl(payload_json)
    except Exception as ex:
        traceback.print_exc()
        return JSON({"error": f"{type(ex).__name__}: {ex}"})

def triage_clinical_exact_v3(p_name, p_age, p_national_id, user_text, p_sbp="", p_dbp="", p_spo2="", p_rr="", p_temp="", p_meds="", p_language="ar", p_hr="", p_eta="", p_gender=""):
    try:
        return _triage_impl(p_name, p_age, p_national_id, user_text, p_sbp, p_dbp, p_spo2, p_rr, p_temp, p_meds, p_language, p_hr, p_eta, p_gender)
    except Exception as ex:
        traceback.print_exc()
        return JSON({"error": f"{type(ex).__name__}: {ex}"})

def clear_case_drafts_v1(draft_id=""):
    try:
        if draft_id:
            DRAFTS.pop(str(draft_id), None)
        return JSON({"ok": True})
    except Exception as ex:
        return JSON({"ok": False, "error": str(ex)})

# ============================================================
# 5. تفريغ الصوت الطبي السريع
# ============================================================
def _transcribe_config(model, thinking=True):
    kwargs = dict(
        system_instruction=(
            "أنت مفرّغ صوتي طبي خبير في مصر (عربي عامي + إنجليزي طبي). فرّغ التسجيل حرفياً بدقة وسرعة دون أي شرح: "
            "1. اكتب الأرقام والقياسات بالأرقام لا بالحروف (مثل 120/80 و 94% و 38.5). "
            "2. اكتب الرقم القومي كـ 14 رقماً متصلاً بدون مسافات إذا نُطق. "
            "3. إذا نُطق اسم دواء إنجليزي بلكنة عربية (مثل كونكور، بلافكس، ماريفان، زاريلتو، أسبرين، انسولين، فينتولين، لازيكس، اوجمنتين)، اكتبه بالإنجليزية الصحيحة مع المادة الفعالة إن أمكن. "
            "4. إذا لم يكن هناك كلام مفهوم أعد نصاً فارغاً."
        ),
        temperature=0.0,
        max_output_tokens=800,
    )
    th = _thinking_cfg(model) if thinking else None
    if th:
        kwargs["thinking_config"] = th
    return types.GenerateContentConfig(**kwargs)

def _parse_text(resp):
    text = (resp.text or "").strip()
    if not text:
        raise ValueError("رد فارغ")
    return text

def _to_wav_bytes(raw, mime):
    ext = "mp4" if "mp4" in mime else ("ogg" if "ogg" in mime else "webm")
    with tempfile.TemporaryDirectory() as td:
        src, dst = os.path.join(td, f"in.{ext}"), os.path.join(td, "out.wav")
        with open(src, "wb") as f:
            f.write(raw)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ar", "16000", "-ac", "1", dst], check=True, capture_output=True, timeout=25)
        with open(dst, "rb") as f:
            return f.read()

def transcribe_audio_v1(b64_audio, mime="audio/webm"):
    try:
        raw = base64.b64decode(b64_audio)
        if len(raw) < 500:
            return JSON({"error": "التسجيل فارغ أو قصير جداً."})
        try:
            wav = _to_wav_bytes(raw, mime or "audio/webm")
            audio_part = types.Part.from_bytes(data=wav, mime_type="audio/wav")
        except Exception as conv_err:
            if "ogg" in (mime or ""):
                audio_part = types.Part.from_bytes(data=raw, mime_type="audio/ogg")
            else:
                return JSON({"error": f"فشل تحويل الصوت (ffmpeg): {conv_err}"})
        text, model_used = generate_with_fallback(
            contents=[audio_part, "فرّغ هذا التسجيل الطبي بدقة (مع كتابة الأدوية الإنجليزية بشكل صحيح وجمع الرقم القومي)."],
            make_config=_transcribe_config,
            parse=_parse_text,
            budget_sec=25,
            rounds=2,
        )
        return JSON({"text": text, "model": model_used})
    except Exception as e:
        return JSON({"error": str(e)})

# ============================================================
# 6a. واجهة المسعف — RELAY (Bridging the Gap Before Arrival)
# ============================================================
APP_HTML = r"""
<style>
  #triageRoot {
    --navy: #0a192f; --navy-mid: #0f2942; --blue: #0284c7; --cyan: #38bdf8; --blue-soft: #e0f2fe; --blue-bg: #f8fafc;
    --ink: #0f172a; --muted: #64748b; --line: #dbe4ef; --surface: #ffffff;
    box-sizing: border-box; max-width: 1020px; margin: 14px auto; padding: clamp(14px, 2.5vw, 26px);
    border: 1px solid var(--line); border-radius: 16px; background: var(--blue-bg); color: var(--ink);
    font-family: system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif; direction: rtl;
    box-shadow: 0 8px 28px rgba(10, 25, 47, 0.08); text-align: initial;
  }
  #triageRoot * { box-sizing: border-box; }
  #triageRoot h3, #triageRoot h4 { color: var(--navy-mid); }
  .topbar { display: flex; justify-content: space-between; align-items: center; gap: 12px; margin-bottom: 16px; padding: 14px 18px; border-radius: 14px; background: linear-gradient(135deg, #0a192f 0%, #0f2942 65%, #0369a1 100%); color: #ffffff; flex-wrap: wrap; box-shadow: 0 4px 14px rgba(10, 25, 47, 0.16); }
  .brand { display: flex; align-items: center; gap: 12px; font-weight: 900; color: #ffffff; font-size: clamp(19px, 2.5vw, 24px); letter-spacing: 0.4px; }
  .brand-mark { width: 46px; height: 46px; border-radius: 12px; background: rgba(56, 189, 248, 0.16); border: 1px solid rgba(56, 189, 248, 0.45); display: grid; place-items: center; font-size: 23px; }
  .brand-tagline { font-size: 12px; color: #7dd3fc; font-weight: 600; display: block; margin-top: 2px; letter-spacing: 0.2px; }
  .er-nav-btn { background: #0284c7; color: #ffffff; border: 1px solid #38bdf8; padding: 8px 14px; border-radius: 9px; font-weight: 750; font-size: 13px; text-decoration: none; display: inline-flex; align-items: center; gap: 6px; }
  .er-nav-btn:hover { background: #0369a1; }
  .workspace-bar { display: flex; justify-content: space-between; align-items: center; gap: 10px; flex-wrap: wrap; background: #ffffff; border: 1px solid var(--line); border-radius: 12px; padding: 10px 14px; margin-bottom: 12px; }
  .section-card { background: var(--surface); border: 1px solid var(--line); border-radius: 14px; padding: clamp(14px, 2vw, 20px); margin: 12px 0; box-shadow: 0 1px 3px rgba(15, 41, 66, 0.03); }
  .section-title { display: flex; align-items: center; justify-content: space-between; gap: 8px; margin: 0 0 14px; padding-bottom: 8px; border-bottom: 1px solid #eef2f6; font-size: 15px; font-weight: 800; color: var(--navy-mid); }
  .patient-inputs { display: grid; grid-template-columns: 1.8fr 0.9fr 1.1fr 1.8fr; gap: 12px; }
  .input-box { min-width: 0; display: flex; flex-direction: column; gap: 6px; }
  .field-head { display: flex; align-items: center; justify-content: space-between; gap: 6px; flex-wrap: wrap; min-height: 20px; }
  .input-box label, .rv-lb { font-size: 12.5px; font-weight: 700; color: #1e293b; margin: 0; }
  .input-box input, .input-box textarea, .rv-in, .input-box select { width: 100%; min-width: 0; padding: 10px 12px; border: 1px solid #cbd5e1; border-radius: 9px; font: inherit; font-size: 14px; background: #fff; color: var(--ink); outline: none; transition: border-color .15s, box-shadow .15s; }
  .input-box input::placeholder, .input-box textarea::placeholder { color: #94a3b8; font-size: 12.5px; }
  .input-box input:focus, .input-box textarea:focus, .rv-in:focus, .input-box select:focus { border-color: var(--blue); box-shadow: 0 0 0 3px rgba(2, 132, 199, 0.14); }
  .vitals-grid { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 12px; }
  .vitals-grid-secondary { display: grid; grid-template-columns: 1fr 1fr 1.6fr; gap: 12px; margin-top: 12px; }
  .manual-badge { display: none; align-items: center; gap: 3px; font-size: 11px; font-weight: 700; color: #0369a1; background: #e0f2fe; border: 1px solid #bae6fd; padding: 1px 7px; border-radius: 999px; white-space: nowrap; }
  .manual-badge.visible { display: inline-flex; }
  .manually-edited { border-color: #0284c7 !important; background: #f0f9ff !important; }
  .auto-filled { background: #eaf7ee !important; border-color: #4b9b67 !important; }
  .map-card { display: flex; flex-direction: column; justify-content: center; padding: 10px 14px; border-radius: 10px; border: 1.5px solid #cbd5e1; background: #f8fafc; min-height: 74px; }
  .map-card-top { display: flex; align-items: center; justify-content: space-between; gap: 6px; font-size: 12px; font-weight: 750; color: #334155; }
  .map-card-val { font-size: 20px; font-weight: 850; margin-top: 4px; color: var(--navy-mid); }
  .map-card-sub { font-size: 11px; color: #64748b; margin-top: 2px; }
  .map-card.st-NA { background: #f8fafc; border-color: #cbd5e1; }
  .map-card.st-NA .map-card-val { color: #64748b; font-size: 13.5px; font-weight: 700; }
  .map-card.st-OK { background: #ecfdf5; border-color: #28a745; }
  .map-card.st-OK .map-card-val { color: #14532d; }
  .map-card.st-WATCH { background: #fefce8; border-color: #eab308; }
  .map-card.st-WATCH .map-card-val { color: #854d0e; }
  .map-card.st-ORANGE { background: #fff7ed; border-color: #e67e22; }
  .map-card.st-ORANGE .map-card-val { color: #9a3412; }
  .map-card.st-RED { background: #fef2f2; border-color: #d9534f; }
  .map-card.st-RED .map-card-val { color: #991b1b; }
  .mic-hero { border: 1px solid #bae6fd; background: linear-gradient(135deg, #f0f9ff, #e0f2fe); border-radius: 14px; padding: 14px 16px; margin: 12px 0; }
  .mic-row { display: flex; align-items: center; justify-content: space-between; gap: 12px; flex-wrap: wrap; }
  .mic-title { font-weight: 800; color: var(--navy-mid); font-size: 15.5px; }
  .mic-sub { font-size: 12px; color: #334155; margin-top: 3px; }
  .mic-btn { display: inline-flex; align-items: center; justify-content: center; gap: 8px; border: 0; background: var(--blue); color: #fff; font-weight: 800; padding: 11px 18px; border-radius: 10px; cursor: pointer; font: inherit; font-size: 14.5px; min-height: 44px; }
  .mic-btn:hover { background: #0369a1; }
  .mic-btn.rec { background: #c62828; animation: pulse 1.2s infinite; }
  .mic-btn:disabled { opacity: .55; cursor: not-allowed; animation: none; }
  .ai-extract-btn { display: inline-flex; align-items: center; gap: 6px; background: #0a192f; color: #fff; border: 0; padding: 7px 13px; border-radius: 8px; font-weight: 700; font-size: 12.5px; cursor: pointer; font-family: inherit; }
  .ai-extract-btn:hover { background: #1e3a5f; }
  .ai-extract-btn:disabled { opacity: .6; cursor: wait; }
  .lang-btn { background: rgba(255,255,255,0.12); color: #ffffff; border: 1px solid rgba(255,255,255,0.3); padding: 8px 13px; border-radius: 9px; font-weight: 700; font-size: 13px; cursor: pointer; }
  .lang-btn:hover { background: rgba(255,255,255,0.22); }
  .lang-btn-light { background: #ffffff; color: var(--navy-mid); border: 1px solid #cbd5e1; padding: 8px 13px; border-radius: 9px; font-weight: 700; font-size: 13px; cursor: pointer; }
  .lang-btn-light:hover { background: #f1f5f9; }
  .btn-reset-case { background: #fff1f2; color: #be123c; border: 1px solid #fecdd3; padding: 8px 14px; border-radius: 9px; font-weight: 800; font-size: 13px; cursor: pointer; font-family: inherit; }
  .btn-reset-case:hover { background: #ffe4e6; border-color: #fda4af; }
  @keyframes pulse { 0%{box-shadow:0 0 0 0 rgba(198,40,40,.45)} 70%{box-shadow:0 0 0 10px rgba(198,40,40,0)} 100%{box-shadow:0 0 0 0 rgba(198,40,40,0)} }
  #waveWrap { display: none; background: #0a192f; border-radius: 10px; padding: 8px 10px; margin: 10px 0 0; }
  #waveCanvas { width: 100%; height: 48px; display: block; }
  #micStatus { color: #e2e8f0; font-size: 12px; text-align: center; margin-top: 4px; }
  .btn-submit { background: #0284c7; color: #fff; border: 0; padding: 13px 16px; font-size: 15px; font-weight: 800; border-radius: 10px; cursor: pointer; width: 100%; font-family: inherit; min-height: 48px; }
  .btn-submit:hover { background: #0369a1; }
  .btn-submit:disabled { background: #94a3b8; cursor: wait; }
  .btn-emergency { background: #b42318; color: #fff; border: 0; padding: 12px 16px; border-radius: 10px; font-weight: 850; font: inherit; cursor: pointer; min-height: 48px; }
  .btn-emergency:hover { background: #921d14; }
  .btn-map { background: #f8fafc; border: 1px dashed #94a3b8; color: #475569; padding: 7px 10px; border-radius: 8px; font-weight: 600; font: inherit; font-size: 11.5px; cursor: default; width: 100%; }
  .eta-chip { padding: 7px 10px; border: 1px solid #bae6fd; background: #f0f9ff; color: #0369a1; border-radius: 8px; cursor: pointer; font-weight: 750; font-size: 12px; }
  .eta-chip:hover { background: #e0f2fe; }
  #autoNote { display: none; background: #edf8f0; color: #24633a; border: 1px solid #b7dfc2; border-radius: 8px; padding: 8px 10px; margin: 10px 0 0; font-size: 12px; }
  .rv-lb { display: block; margin-bottom: 4px; }
  .rv-in { margin: 4px 0 10px; }
  .patients-table { width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 12px; }
  .patients-table th, .patients-table td { border: 1px solid #dbe4ef; padding: 8px; }
  .patients-table th { background: #f1f5f9; color: var(--navy-mid); position: sticky; top: 0; }
  .disclaimer { margin-top: 14px; font-size: 11px; line-height: 1.6; color: #64748b; }
  .modal-backdrop { display: none; position: fixed; inset: 0; background: rgba(10, 25, 47, 0.6); z-index: 9999; place-items: center; padding: 16px; }
  .modal-backdrop.open { display: grid; }
  .modal-dialog { background: #ffffff; border: 1px solid var(--line); border-radius: 14px; max-width: 440px; width: 100%; padding: 20px; box-shadow: 0 20px 40px rgba(10, 25, 47, 0.25); }
  .modal-title { margin: 0 0 8px; font-size: 17px; font-weight: 800; color: var(--navy-mid); }
  .modal-body { margin: 0 0 18px; font-size: 14px; color: #334155; line-height: 1.55; }
  .modal-actions { display: flex; justify-content: flex-end; gap: 10px; }
  .btn-modal-cancel { background: #f1f5f9; color: #334155; border: 1px solid #cbd5e1; padding: 9px 16px; border-radius: 9px; font-weight: 700; cursor: pointer; font-family: inherit; }
  .btn-modal-confirm { background: #d9534f; color: #ffffff; border: 0; padding: 9px 18px; border-radius: 9px; font-weight: 800; cursor: pointer; font-family: inherit; }
  @media(max-width: 760px) { .patient-inputs, .vitals-grid, .vitals-grid-secondary { grid-template-columns: 1fr 1fr; } .mic-row { align-items: stretch; } .mic-btn { width: 100%; } }
  @media(max-width: 480px) { .patient-inputs, .vitals-grid, .vitals-grid-secondary { grid-template-columns: 1fr; } }
</style>

<div class="triage-card" id="triageRoot">
  <div class="topbar">
    <div class="brand">
      <span class="brand-mark">🚑</span>
      <div>
        <span>RELAY</span>
        <span class="brand-tagline">Bridging the Gap Before Arrival &nbsp;|&nbsp; Pre-Hospital EMS Handover</span>
      </div>
    </div>
    <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
      <a href="__ER_URL__" target="_blank" rel="noopener" class="er-nav-btn">🏥 شاشة استقبال الطوارئ (ER Dashboard)</a>
      <button type="button" id="langToggle" class="lang-btn">English</button>
    </div>
  </div>

  <section id="loginScreen" class="section-card" style="max-width:460px;margin:24px auto;">
    <h3 style="margin:0 0 6px;font-size:20px;">دخول طاقم الإسعاف — RELAY</h3>
    <p style="margin:0 0 16px;color:#64748b;font-size:13px;">أدخل كود المسعف والرقم السري مرة واحدة لبدء تسجيل الحالات.</p>
    <div class="input-box" style="margin-bottom:12px;">
      <label for="loginCode">كود المسعف</label>
      <input id="loginCode" autocomplete="username" placeholder="أدخل كود المسعف" autocapitalize="characters">
    </div>
    <div class="input-box" style="margin-bottom:14px;">
      <label for="loginPin">الرقم السري (PIN)</label>
      <input id="loginPin" type="password" inputmode="numeric" autocomplete="current-password" placeholder="••••">
    </div>
    <button id="loginBtn" type="button" class="btn-submit">دخول إلى النظام</button>
    <div id="loginStatus" role="status" style="font-size:12px;color:#b42318;margin-top:10px;min-height:18px;"></div>
  </section>

  <div id="appWorkspace" style="display:none;">
    <div class="workspace-bar">
      <div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap;">
        <h3 style="margin:0;font-size:17px;color:var(--navy-mid);">🚑 تسجيل حالة جديدة</h3>
        <span id="activeParamedicLabel" style="font-size:12.5px;color:#475569;font-weight:600;">المسعف: —</span>
      </div>
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <button type="button" id="clearCaseBtn" class="btn-reset-case">🗑️ مسح بيانات الحالة</button>
        <button type="button" id="logoutBtn" class="lang-btn-light">تسجيل الخروج</button>
      </div>
    </div>

    <div style="background:#fffbeb;border:1px solid #f59e0b;color:#78350f;border-radius:10px;padding:10px 12px;margin-bottom:10px;font-size:12.5px;">
      <b>نسخة تجريبية — ليست أداة قرار سريري.</b> الذكاء الاصطناعي يقترح <b>مسودة</b> فقط وقد يخطئ، والقرار والتأكيد للمسعف المسؤول عن المريض.
    </div>
    <div id="sysStatus" style="font-size:12px;color:#64748b;margin-bottom:8px;">⏳ فحص الاتصال بالبايثون...</div>
    <div id="appError" style="display:none;background:#fdecea;color:#b71c1c;border:1px solid #f5c2c0;border-radius:8px;padding:9px 12px;margin-bottom:10px;font-size:13px;"></div>

    <div class="mic-hero">
      <div class="mic-row">
        <div>
          <div class="mic-title">🎙️ التسجيل الصوتي المباشر + التوزيع الذكي بالذكاء الاصطناعي</div>
          <div class="mic-sub">تحدث بصورة طبيعية بالعربية أو الإنجليزية؛ يستخرج النظام الاسم، العمر، النوع، الرقم القومي، العلامات الحيوية، والأدوية تلقائياً مع حفظ النص الحرفي كاملاً.</div>
        </div>
        <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
          <select id="micLang" title="لغة الكلام" style="padding:9px 10px;border-radius:9px;border:1px solid #bae6fd;background:#fff;font:inherit;font-size:12.5px;">
            <option value="ar-EG">عربي (مصر)</option>
            <option value="en-US">English</option>
          </select>
          <button type="button" class="mic-btn" id="micBtn">🎤 ابدأ التسجيل</button>
        </div>
      </div>
      <div id="micError" style="display:none;background:#fdecea;color:#b71c1c;border:1px solid #f5c2c0;border-radius:8px;padding:8px 10px;margin-top:8px;font-size:13px;"></div>
      <div id="waveWrap"><canvas id="waveCanvas"></canvas><div id="micStatus">جاري التسجيل...</div></div>
    </div>

    <section class="section-card">
      <div class="section-title"><span>👤 1. بيانات تعريف المريض</span></div>
      <div class="patient-inputs">
        <div class="input-box">
          <div class="field-head"><label for="pName">اسم المريض:</label><span class="manual-badge" id="badge_pName">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="pName" placeholder="أدخل الاسم الثلاثي" autocomplete="off">
        </div>
        <div class="input-box">
          <div class="field-head"><label for="pAge">العمر (سنة):</label><span class="manual-badge" id="badge_pAge">✎ تم التعديل يدويًا</span></div>
          <input type="number" id="pAge" placeholder="العمر (0 للرضع)" min="0" max="130" autocomplete="off">
        </div>
        <div class="input-box">
          <div class="field-head"><label for="pGender">النوع (Gender):</label><span class="manual-badge" id="badge_pGender">✎ تم التعديل يدويًا</span></div>
          <select id="pGender">
            <option value="">غير محدد</option>
            <option value="Male">ذكر (Male)</option>
            <option value="Female">أنثى (Female)</option>
          </select>
        </div>
        <div class="input-box">
          <div class="field-head"><label for="pNationalId">الرقم القومي (14 رقماً):</label><span class="manual-badge" id="badge_pNationalId">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="pNationalId" placeholder="14 رقماً دون حروف" maxlength="14" inputmode="numeric" pattern="[0-9]*" autocomplete="off">
        </div>
      </div>
    </section>

    <section class="section-card">
      <div class="section-title"><span>🩺 2. العلامات الحيوية ومتوسط الضغط الشرياني (MAP)</span></div>
      <div class="vitals-grid">
        <div class="input-box">
          <div class="field-head"><label for="vSys">🩸 الضغط الانقباضي (SBP):</label><span class="manual-badge" id="badge_vSys">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="vSys" placeholder="مم زئبق (40–300)" maxlength="3" inputmode="numeric" autocomplete="off">
        </div>
        <div class="input-box">
          <div class="field-head"><label for="vDia">🩸 الضغط الانبساطي (DBP):</label><span class="manual-badge" id="badge_vDia">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="vDia" placeholder="مم زئبق (20–200)" maxlength="3" inputmode="numeric" autocomplete="off">
        </div>
        <div class="map-card st-NA" id="mapDisplayCard" role="region" aria-live="polite">
          <div class="map-card-top"><span id="mapTitleLabel">متوسط الضغط الشرياني (MAP)</span><span id="mapStatusIcon">⚪</span></div>
          <div class="map-card-val" id="mapValueText">غير متاح — أدخل الانقباضي والانبساطي</div>
          <div class="map-card-sub" id="mapSubText">تقديري: DBP + (SBP − DBP) / 3</div>
        </div>
        <div class="input-box">
          <div class="field-head"><label for="vHR">💓 النبض HR (نبضة/د):</label><span class="manual-badge" id="badge_vHR">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="vHR" placeholder="نبضة/د (20–260)" maxlength="3" inputmode="numeric" autocomplete="off">
        </div>
      </div>
      <div class="vitals-grid-secondary">
        <div class="input-box">
          <div class="field-head"><label for="vSpo2">🫁 الأكسجين SpO₂ (%):</label><span class="manual-badge" id="badge_vSpo2">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="vSpo2" placeholder="النسبة % (50–100)" maxlength="3" inputmode="numeric" autocomplete="off">
        </div>
        <div class="input-box">
          <div class="field-head"><label for="vRR">🫁 معدل التنفس RR (نفس/د):</label><span class="manual-badge" id="badge_vRR">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="vRR" placeholder="نفس/د (4–80)" maxlength="2" inputmode="numeric" autocomplete="off">
        </div>
        <div class="input-box">
          <div class="field-head"><label for="vTemp" id="vTempLabel">🌡️ الحرارة (°م):</label><span class="manual-badge" id="badge_vTemp">✎ تم التعديل يدويًا</span></div>
          <input type="text" id="vTemp" placeholder="درجة مئوية (28–45)" maxlength="5" inputmode="decimal" autocomplete="off">
        </div>
      </div>
      <div id="autoNote"></div>
    </section>

    <section class="section-card">
      <div class="section-title"><span>🧠 3. تقييم مستوى الوعي (AVPU & GCS)</span></div>
      <div class="vitals-grid" style="grid-template-columns:repeat(2,minmax(0,1fr));">
        <div class="input-box">
          <div class="field-head"><label for="vConsciousness">AVPU – مستوى الاستجابة:</label><span class="manual-badge" id="badge_vConsciousness">✎ تم التعديل يدويًا</span></div>
          <select id="vConsciousness">
            <option value="">لم يُقيّم</option>
            <option value="A">A – Alert / واعٍ</option>
            <option value="V">V – Responds to Voice / يستجيب للصوت</option>
            <option value="P">P – Responds to Pain / يستجيب للألم</option>
            <option value="U">U – Unresponsive / لا يستجيب</option>
          </select>
        </div>
        <div class="input-box">
          <div class="field-head"><label for="vGCS">مقياس غلاسكو للغيبوبة GCS (3–15):</label><span class="manual-badge" id="badge_vGCS">✎ تم التعديل يدويًا</span></div>
          <input type="number" id="vGCS" min="3" max="15" placeholder="أدخل الدرجة (3–15)" inputmode="numeric">
        </div>
      </div>
    </section>

    <section class="section-card">
      <div class="section-title">
        <span>📝 4. الشكوى الرئيسية والأدوية الحالية</span>
        <button type="button" id="aiExtractBtn" class="ai-extract-btn" title="توزيع ذكي للنص المكتوب على جميع الخانات">✨ استخراج ذكي من النص</button>
      </div>
      <div class="input-box" style="margin-bottom:12px;">
        <div class="field-head"><label for="complaintText">شكوى المريض الرئيسية (السجل الحرفي الكامل):</label><span class="manual-badge" id="badge_complaintText">✎ تم التعديل يدويًا</span></div>
        <textarea id="complaintText" placeholder="اكتب الشكوى السريرية أو استخدم التسجيل الصوتي بالأعلى (يُحفظ النص هنا كاملاً دون حذف أي حرف)..." style="width:100%;height:92px;resize:vertical;"></textarea>
        <div id="asrNote" style="display:none;margin-top:4px;font-size:12px;color:#8a6d00;">⚠️ النص منقول من الصوت وقد يحتوي أخطاء — راجعه (خاصةً الأرقام وأسماء الأدوية) قبل المتابعة.</div>
      </div>
      <div class="input-box">
        <div class="field-head"><label for="vMeds">💊 الأدوية التي يتناولها المريض:</label><span class="manual-badge" id="badge_vMeds">✎ تم التعديل يدويًا</span></div>
        <input type="text" id="vMeds" placeholder="اسم تجاري أو مادة فعالة أو فئة علاجية، افصل بفاصلة" maxlength="300" autocomplete="off">
      </div>
    </section>

    <section class="section-card">
      <div class="section-title"><span>⏱️ 5. وقت الوصول المتوقع والمرفقات الإضافية</span></div>
      <div class="vitals-grid" style="grid-template-columns:1.3fr 1fr;">
        <div class="input-box">
          <div class="field-head"><label for="vETA">⏱️ الوصول المتوقع للمستشفى (ETA بالدقائق):</label><span class="manual-badge" id="badge_vETA">✎ تم التعديل يدويًا</span></div>
          <div style="display:flex;gap:6px;align-items:center;">
            <input type="text" id="vETA" placeholder="الوقت بالدقائق (1–180)" maxlength="3" inputmode="numeric" autocomplete="off" style="flex:1;">
            <button type="button" class="eta-chip" data-eta="5">5د</button>
            <button type="button" class="eta-chip" data-eta="10">10د</button>
            <button type="button" class="eta-chip" data-eta="15">15د</button>
          </div>
          <div style="margin-top:6px;"><button type="button" class="btn-map" id="mapsEtaDemo" disabled>📍 حساب الوصول عبر Google Maps — قريباً</button></div>
        </div>
        <div class="input-box">
          <label for="historyFile">📎 التاريخ المرضي / التقارير الطبية (اختياري):</label>
          <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
            <input type="file" id="historyFile" accept=".pdf,.png,.jpg,.jpeg,.webp,.txt,.doc,.docx" style="max-width:100%;">
            <span id="historyFileName" style="font-size:12px;color:#52657d;">لم يتم اختيار ملف</span>
          </div>
          <div style="font-size:11px;color:#64748b;margin-top:3px;">إرفاق مبدئي للملف داخل الواجهة فقط؛ لا يُرسل إلى Gemini ولا يدخل في التقييم السريري في هذه النسخة.</div>
        </div>
      </div>
    </section>

    <div style="display:grid;grid-template-columns:minmax(0,1fr) auto;gap:10px;align-items:stretch;margin-top:12px;">
      <button id="submitBtn" type="button" class="btn-submit" style="margin:0;">📝 إنشاء مسودة للمراجعة</button>
      <button id="emergencyBtn" type="button" class="btn-emergency">🚨 EMERGENCY ALERT</button>
    </div>
    <div id="emergencyStatus" role="status" style="font-size:12px;margin-top:6px;color:#9b1c1c;"></div>
    <div id="triageLoading" style="display:none;margin-top:8px;color:#0284c7;font-weight:bold;text-align:center;"></div>

    <div id="resultCard" style="display:none;margin-top:16px;padding:16px;border-radius:12px;color:white;"></div>
    <div id="savedBanner" style="display:none;margin-top:12px;padding:10px 12px;border-radius:8px;font-size:13px;"></div>

    <div id="reviewPanel" style="display:none;margin-top:12px;padding:16px;border:2px solid #e8bd57;border-radius:12px;background:#fffaf0;color:#222;">
      <h4 style="margin:0 0 6px;color:#654d03;">✋ مراجعة قبل الحفظ والإرسال</h4>
      <div id="reviewModelNote" style="font-size:12px;margin-bottom:6px;"></div>
      <div id="reviewStale" style="display:none;background:#fdecea;color:#b71c1c;border:1px solid #f5c2c0;border-radius:6px;padding:6px 10px;margin-bottom:8px;font-size:13px;">⚠️ تغيّرت المدخلات بعد إنشاء المسودة. أنشئ مسودة جديدة قبل التأكيد.</div>
      <div id="reviewSnapshot" style="font-size:12px;background:#fff;border:1px dashed #d9c27a;border-radius:6px;padding:8px 10px;margin-bottom:8px;"></div>

      <label class="rv-lb" for="reviewSummary">الملخص المقترح (قابل للتعديل)</label>
      <textarea id="reviewSummary" class="rv-in" style="min-height:52px;"></textarea>

      <div style="display:flex;gap:10px;flex-wrap:wrap;">
        <div style="flex:1;min-width:120px;">
          <label class="rv-lb" for="reviewTier">الفئة</label>
          <select id="reviewTier" class="rv-in">
            <option value="RED">🔴 أحمر (9-10)</option>
            <option value="ORANGE">🟠 برتقالي (5-8)</option>
            <option value="GREEN">🟢 أخضر (1-4)</option>
          </select>
        </div>
        <div style="flex:1;min-width:90px;">
          <label class="rv-lb" for="reviewScore">الدرجة (1-10)</label>
          <input id="reviewScore" class="rv-in" type="number" min="1" max="10">
        </div>
        <div style="flex:2;min-width:220px;">
          <label class="rv-lb" for="reviewDepartment">القسم</label>
          <select id="reviewDepartment" class="rv-in"></select>
        </div>
      </div>
      <input id="reviewDeptOther" class="rv-in" type="text" maxlength="80" placeholder="حدّد القسم (للحالات الاستثنائية فقط)" style="display:none;">

      <div id="reviewOverrideWrap" style="display:none;">
        <label class="rv-lb" for="reviewOverride">⚠️ سبب التخفيض عن الحد الذي تفرضه العلامات الحيوية (إلزامي)</label>
        <input id="reviewOverride" class="rv-in" type="text" maxlength="200">
      </div>

      <div class="rv-lb">التجهيزات المقترحة</div>
      <div id="reviewChips" style="font-size:13px;line-height:1.7;background:#fff;border:1px solid #eadfb8;border-radius:6px;padding:6px 10px;margin:4px 0 6px;"></div>
      <details id="reviewDetails" style="margin-bottom:10px;">
        <summary style="cursor:pointer;font-size:13px;color:#654d03;font-weight:bold;">تعديل التجهيزات أو إضافة ملاحظة (اختياري)</summary>
        <div id="reviewPrep"></div>
        <label class="rv-lb" for="reviewOtherNote">ملاحظة حرة (للحالات الاستثنائية فقط)</label>
        <input id="reviewOtherNote" class="rv-in" type="text" maxlength="200">
      </details>

      <input id="reviewReviewer" type="hidden" value="">
      <label style="display:flex;gap:8px;align-items:flex-start;font-weight:bold;margin:6px 0 10px;color:#654d03;font-size:13px;cursor:pointer;">
        <input type="checkbox" id="reviewConfirmed" style="margin-top:3px;">
        راجعت الشكوى والعلامات الحيوية والاقتراحات، وأتحمل مسؤولية تأكيد بيانات التسليم وفق بروتوكول المنشأة.
      </label>
      <button id="confirmSaveBtn" type="button" class="btn-submit" style="background:#0a192f;margin-top:0;">✅ تأكيد وحفظ وإرسال التنبيه</button>
      <button id="cancelReviewBtn" type="button" style="width:100%;margin-top:8px;padding:9px;border:1px solid #94a3b8;border-radius:8px;background:#fff;cursor:pointer;font-family:inherit;">إلغاء المسودة</button>
      <div id="saveStatus" role="status" style="margin-top:8px;font-size:13px;color:#b71c1c;"></div>
    </div>

    <div style="margin-top:22px;border-top:2px solid #dbe4ef;padding-top:12px;">
      <h4 style="margin:0 0 8px 0;color:var(--navy-mid);">📋 الحالات المؤكَّدة بشرياً والمحفوظة (RELAY Log):</h4>
      <div style="max-height:260px;overflow:auto;">
        <table class="patients-table">
          <thead>
            <tr><th>الوقت</th><th>رقم المريض</th><th>الاسم</th><th>العمر</th><th>النوع</th><th>الرقم القومي</th><th>الملخص</th><th>الفلاج</th><th>الخطورة</th><th>القسم</th><th>العلامات الحيوية</th><th>النموذج</th><th>المراجع</th></tr>
          </thead>
          <tbody id="patientsListBody">__INITIAL_ROWS__</tbody>
        </table>
      </div>
    </div>
  </div>

  <div class="modal-backdrop" id="resetModalBackdrop" role="dialog" aria-modal="true">
    <div class="modal-dialog">
      <h4 class="modal-title" id="resetModalTitle">🗑️ مسح بيانات الحالة</h4>
      <p class="modal-body" id="resetModalBody">هل أنت متأكد من مسح جميع بيانات الحالة الحالية؟ لا يمكن التراجع عن هذا الإجراء.</p>
      <div class="modal-actions">
        <button type="button" class="btn-modal-cancel" id="resetCancelBtn">إلغاء</button>
        <button type="button" class="btn-modal-confirm" id="resetConfirmBtn">تأكيد المسح</button>
      </div>
    </div>
  </div>
  <div class="disclaimer">⚠️ نموذج أولي غير معتمد (RELAY Prototype): المخرجات مسودات لا تُعد تشخيصاً ولا قرار فرز. الشكوى والعلامات الحيوية والعمر والنوع تُرسل إلى خدمة Gemini (الاسم والرقم القومي يُحجبان تلقائياً).</div>
</div>

<script>
(function () {
  const $ = id => document.getElementById(id);
  const CATALOG = __CATALOG_JSON__;
  const CALL_TOKEN = "__CALL_TOKEN__";

  const ALL_EDITABLE_FIELDS = ['pName', 'pAge', 'pGender', 'pNationalId', 'vSys', 'vDia', 'vHR', 'vRR', 'vSpo2', 'vTemp', 'vConsciousness', 'vGCS', 'vMeds', 'complaintText', 'vETA'];
  const autoFlag = {}, manualFlag = {};
  ALL_EDITABLE_FIELDS.forEach(id => { autoFlag[id] = false; manualFlag[id] = false; });

  const LANG_KEY = 'triageUiLanguage';
  const translations = {
    '✎ تم التعديل يدويًا': '✎ Manually edited', '🗑️ مسح بيانات الحالة': '🗑️ Clear Patient Case',
    '✨ استخراج ذكي من النص': '✨ Smart AI Extract', '🏥 شاشة استقبال الطوارئ (ER Dashboard)': '🏥 ER Hospital Live Dashboard',
    'متوسط الضغط الشرياني (MAP)': 'Mean Arterial Pressure (MAP)', 'غير متاح — أدخل الانقباضي والانبساطي': 'Unavailable — enter SBP & DBP',
    '👤 1. بيانات تعريف المريض': '👤 1. Patient Identification', '🩺 2. العلامات الحيوية ومتوسط الضغط الشرياني (MAP)': '🩺 2. Main Vital Signs & MAP',
    '🧠 3. تقييم مستوى الوعي (AVPU & GCS)': '🧠 3. Consciousness Assessment (AVPU & GCS)', '📝 4. الشكوى الرئيسية والأدوية الحالية': '📝 4. Chief Complaint & Medications',
    '⏱️ 5. وقت الوصول المتوقع والمرفقات الإضافية': '⏱️ 5. Estimated Arrival (ETA) & Attachments', 'اسم المريض:': 'Patient name:',
    'العمر (سنة):': 'Age (years):', 'النوع (Gender):': 'Gender:', 'الرقم القومي (14 رقماً):': 'National ID (14 digits):',
    '🩸 الضغط الانقباضي (SBP):': '🩸 Systolic BP (SBP):', '🩸 الضغط الانبساطي (DBP):': '🩸 Diastolic BP (DBP):',
    '💓 النبض HR (نبضة/د):': '💓 Heart rate HR (bpm):', '🫁 الأكسجين SpO₂ (%):': '🫁 Oxygen saturation SpO₂ (%):',
    '🫁 معدل التنفس RR (نفس/د):': '🫁 Respiratory rate RR (breaths/min):', '💊 الأدوية التي يتناولها المريض:': '💊 Current Medications:',
    'شكوى المريض الرئيسية (السجل الحرفي الكامل):': 'Chief Complaint (Full Raw Transcript):',
    '📝 إنشاء مسودة للمراجعة': '📝 Generate Draft for Review', '✅ تأكيد وحفظ وإرسال التنبيه': '✅ Confirm, save, and send alert',
    'إلغاء المسودة': 'Cancel draft', 'دخول طاقم الإسعاف — RELAY': 'RELAY — EMS Team Sign In', 'دخول إلى النظام': 'Sign In',
    '🚑 تسجيل حالة جديدة': '🚑 New Patient Handover', 'تسجيل الخروج': 'Sign Out', '🎤 ابدأ التسجيل': '🎤 Start Recording'
  };

  const originalText = new WeakMap(), originalAttrs = new WeakMap();
  function translateString(value, toEnglish) {
    let v = String(value);
    if (!toEnglish) return v;
    for (const k of Object.keys(translations).sort((a,b)=>b.length-a.length)) v = v.split(k).join(translations[k]);
    return v;
  }
  function translateTree(root, toEnglish) {
    const walker = document.createTreeWalker(root, NodeFilter.SHOW_TEXT);
    let node;
    while ((node = walker.nextNode())) {
      if (!originalText.has(node)) originalText.set(node, node.nodeValue);
      const nextText = translateString(originalText.get(node), toEnglish);
      if (node.nodeValue !== nextText) node.nodeValue = nextText;
    }
    for (const el of [root, ...(root.querySelectorAll ? Array.from(root.querySelectorAll('*')) : [])]) {
      for (const attr of ['placeholder','title','aria-label']) {
        if (!el.hasAttribute || !el.hasAttribute(attr)) continue;
        let saved = originalAttrs.get(el);
        if (!saved) { saved = {}; originalAttrs.set(el,saved); }
        if (!(attr in saved)) saved[attr] = el.getAttribute(attr);
        el.setAttribute(attr, translateString(saved[attr], toEnglish));
      }
    }
  }

  let uiLanguage = 'ar';
  function refreshManualBadges() {
    const labelText = uiLanguage === 'en' ? '✎ Manually edited' : '✎ تم التعديل يدويًا';
    ALL_EDITABLE_FIELDS.forEach(id => {
      const badge = $('badge_' + id), el = $(id);
      if (!badge || !el) return;
      badge.textContent = labelText;
      if (manualFlag[id]) { badge.classList.add('visible'); el.classList.add('manually-edited'); el.classList.remove('auto-filled'); }
      else { badge.classList.remove('visible'); el.classList.remove('manually-edited'); if (autoFlag[id]) el.classList.add('auto-filled'); }
    });
  }
  function markFieldManuallyEdited(id) { manualFlag[id] = true; autoFlag[id] = false; refreshManualBadges(); updateAutoNote(); }

  function computeDeterministicMAP(sbpRaw, dbpRaw) {
    const sStr = String(sbpRaw || '').trim(), dStr = String(dbpRaw || '').trim();
    if (!sStr || !dStr) return { available: false, reason: 'missing' };
    if (!/^\d{2,3}$/.test(sStr) || !/^\d{2,3}$/.test(dStr)) return { available: false, reason: 'invalid' };
    const sbp = Number(sStr), dbp = Number(dStr);
    if (sbp < 40 || sbp > 300 || dbp < 20 || dbp > 200 || sbp <= dbp || (sbp - dbp) < 5) return { available: false, reason: 'implausible' };
    const exact = dbp + (sbp - dbp) / 3, rounded = Math.round(exact);
    let status = rounded < 60 ? 'RED' : ((rounded < 65 || rounded >= 135) ? 'ORANGE' : ((rounded < 70 || rounded >= 110) ? 'WATCH' : 'OK'));
    return { available: true, rounded, exact: Math.round(exact * 100) / 100, status };
  }

  const MAP_ICONS = { NA: '⚪', OK: '🟢', WATCH: '🟡', ORANGE: '🟠', RED: '🔴' };
  function updateMapDisplay() {
    const card = $('mapDisplayCard'), titleEl = $('mapTitleLabel'), iconEl = $('mapStatusIcon'), valEl = $('mapValueText'), subEl = $('mapSubText');
    if (!card || !valEl) return;
    const res = computeDeterministicMAP($('vSys').value, $('vDia').value);
    card.className = 'map-card st-' + (res.available ? res.status : 'NA');
    titleEl.textContent = uiLanguage === 'en' ? 'Mean Arterial Pressure (MAP)' : 'متوسط الضغط الشرياني (MAP)';
    iconEl.textContent = MAP_ICONS[res.available ? res.status : 'NA'];
    if (!res.available) {
      valEl.textContent = res.reason === 'missing' ? (uiLanguage === 'en' ? 'Unavailable — enter SBP & DBP' : 'غير متاح — أدخل الانقباضي والانبساطي') : 'غير متاح — قيمة غير صالحة';
      subEl.textContent = 'تقديري: DBP + (SBP − DBP) / 3';
      return;
    }
    valEl.textContent = res.rounded + (uiLanguage === 'en' ? ' mmHg' : ' مم زئبق');
    subEl.textContent = res.status === 'RED' ? 'منخفض بشدة — قيّم التروية فوراً' : (res.status === 'ORANGE' ? 'خارج النطاق المعتاد — قيّم التروية' : 'تقديري غير باضع: DBP + (SBP − DBP) / 3');
  }

  function setUiLanguage(lang) {
    const prevLang = uiLanguage;
    uiLanguage = lang === 'en' ? 'en' : 'ar';
    const root = $('triageRoot'), tempInput = $('vTemp');
    if (tempInput && tempInput.value.trim() && prevLang !== uiLanguage) {
      const current = Number(tempInput.value);
      if (Number.isFinite(current)) tempInput.value = (Math.round((uiLanguage === 'en' ? (current * 9 / 5 + 32) : ((current - 32) * 5 / 9)) * 10) / 10).toString();
    }
    root.setAttribute('dir', uiLanguage === 'en' ? 'ltr' : 'rtl');
    document.documentElement.dir = uiLanguage === 'en' ? 'ltr' : 'rtl';
    translateTree(root, uiLanguage === 'en');
    const tempLabel = $('vTempLabel');
    if (tempLabel) tempLabel.textContent = uiLanguage === 'en' ? '🌡️ Temperature (°F):' : '🌡️ الحرارة (°م):';
    $('langToggle').textContent = uiLanguage === 'en' ? 'العربية' : 'English';
    $('micLang').value = uiLanguage === 'en' ? 'en-US' : 'ar-EG';
    refreshManualBadges(); updateMapDisplay(); updateAutoNote();
    try { localStorage.setItem(LANG_KEY, uiLanguage); } catch(e) {}
  }

  $('langToggle').addEventListener('click', () => setUiLanguage(uiLanguage === 'ar' ? 'en' : 'ar'));
  try { uiLanguage = localStorage.getItem(LANG_KEY) === 'en' ? 'en' : 'ar'; } catch(e) {}

  const esc = s => String(s == null ? "" : s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
  const rawCall = async (name, args) => {
    if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {
      const res = await google.colab.kernel.invokeFunction(name, args, {});
      return res.data['application/json'];
    }
    const r = await fetch('call', { method: 'POST', headers: { 'Content-Type': 'application/json', 'X-Token': CALL_TOKEN }, body: JSON.stringify({ name: name, args: args }) });
    if (!r.ok) throw new Error('HTTP ' + r.status);
    return await r.json();
  };
  const callPy = async (name, args, timeoutMs) => {
    let timer;
    const timeout = new Promise((_, rej) => { timer = setTimeout(() => rej(new Error('انتهت مهلة الرد من البايثون.')), timeoutMs || 60000); });
    try { return await Promise.race([rawCall(name, args), timeout]); } finally { clearTimeout(timer); }
  };
  const appErr = msg => { const b = $('appError'); b.style.display = 'block'; b.textContent = '⚠️ ' + msg; };
  const clearAppErr = () => { const b = $('appError'); b.style.display = 'none'; b.textContent = ''; };

  (async () => {
    const el = $('sysStatus');
    try {
      const d = await callPy('notebook.status_v1', [], 10000);
      el.textContent = '✅ RELAY متصل | النماذج: ' + d.models.join(' ← ') + ' | تليجرام: ' + d.telegram + ' | الإصدار: ' + d.version;
      el.style.color = '#15803d';
    } catch (e) {
      el.textContent = '❌ الواجهة غير متصلة بالبايثون. أعد تشغيل خلية الكود.';
      el.style.color = '#c62828';
    }
  })();

  const nid = $('pNationalId'), AR = '٠١٢٣٤٥٦٧٨٩', PR = '۰۱۲۳۴۵۶۷۸۹';
  const toAsciiDigits = s => String(s || '').replace(/[٠-٩]/g, d => AR.indexOf(d)).replace(/[۰-۹]/g, d => PR.indexOf(d));

  nid.addEventListener('input', function () { this.value = toAsciiDigits(this.value).replace(/[^0-9]/g, '').slice(0, 14); });
  $('vSys').addEventListener('input', function () {
    const raw = toAsciiDigits(this.value), m = /^\s*(\d{2,3})\s*[\/\\\\]\s*(\d{2,3})\s*$/.exec(raw);
    if (m) { this.value = m[1].slice(0, 3); $('vDia').value = m[2].slice(0, 3); markFieldManuallyEdited('vSys'); markFieldManuallyEdited('vDia'); updateMapDisplay(); return; }
    this.value = raw.replace(/[^0-9]/g, '').slice(0, 3); updateMapDisplay();
  });
  $('vDia').addEventListener('input', function () { this.value = toAsciiDigits(this.value).replace(/[^0-9]/g, '').slice(0, 3); updateMapDisplay(); });
  ['vSpo2', 'vHR', 'vETA'].forEach(id => $(id).addEventListener('input', function () { this.value = toAsciiDigits(this.value).replace(/[^0-9]/g, '').slice(0, 3); }));
  $('vRR').addEventListener('input', function () { this.value = toAsciiDigits(this.value).replace(/[^0-9]/g, '').slice(0, 2); });
  $('vTemp').addEventListener('input', function () {
    let v = toAsciiDigits(this.value).replace(/[٫,]/g, '.').replace(/[^0-9.]/g, '');
    const i = v.indexOf('.'); if (i !== -1) v = v.slice(0, i + 1) + v.slice(i + 1).replace(/\./g, '');
    this.value = v.slice(0, 5);
  });

  ALL_EDITABLE_FIELDS.forEach(id => {
    const el = $(id); if (!el) return;
    el.addEventListener(el.tagName === 'SELECT' ? 'change' : 'input', () => {
      markFieldManuallyEdited(id);
      if (id === 'vSys' || id === 'vDia') updateMapDisplay();
      markStale();
    });
  });

  function updateAutoNote() {
    const note = $('autoNote');
    const names = { pName: 'الاسم', pAge: 'العمر', pGender: 'النوع', pNationalId: 'الرقم القومي', vSys: 'SBP', vDia: 'DBP', vSpo2: 'SpO₂', vRR: 'RR', vTemp: 'الحرارة', vHR: 'HR', vETA: 'ETA', vMeds: 'الأدوية', vConsciousness: 'AVPU', vGCS: 'GCS' };
    const active = Object.keys(autoFlag).filter(k => autoFlag[k] && !manualFlag[k]);
    if (!active.length) { note.style.display = 'none'; return; }
    note.style.display = 'block';
    note.textContent = '✨ تم ملء (' + active.map(k => names[k] + (k === 'pNationalId' ? '' : ': ' + $(k).value)).join(' | ') + ') تلقائياً — راجعها قبل الإرسال.';
  }

  function applyAutoField(id, val) {
    if (manualFlag[id]) return;
    const el = $(id);
    if (el && val != null && val !== '') { el.value = String(val); autoFlag[id] = true; el.classList.add('auto-filled'); }
  }

  let aiExtractInFlight = false;
  async function runSmartAiExtraction(sourceText) {
    const text = String(sourceText || $('complaintText').value || '').trim();
    if (!text || text.length < 3 || aiExtractInFlight) return;
    aiExtractInFlight = true;
    const btn = $('aiExtractBtn'), origTxt = btn ? btn.textContent : '';
    if (btn) { btn.disabled = true; btn.textContent = '⏳ جاري التوزيع الذكي...'; }
    const note = $('autoNote'); note.style.display = 'block';
    note.textContent = '🧠 جاري التحليل الذكي لاستخراج الاسم والنوع والرقم القومي وتصحيح أسماء الأدوية...';
    try {
      const res = await callPy('notebook.extract_fields_v1', [text, uiLanguage], 15000);
      if (res && res.ok && res.fields) {
        const f = res.fields;
        applyAutoField('pName', f.patient_name);
        applyAutoField('pAge', f.age != null ? String(f.age) : null);
        applyAutoField('pGender', f.gender);
        applyAutoField('pNationalId', f.national_id);
        applyAutoField('vSys', f.sbp);
        applyAutoField('vDia', f.dbp);
        applyAutoField('vHR', f.hr);
        applyAutoField('vSpo2', f.spo2);
        applyAutoField('vRR', f.rr);
        applyAutoField('vTemp', f.temp);
        applyAutoField('vConsciousness', f.avpu);
        applyAutoField('vGCS', f.gcs);
        applyAutoField('vMeds', f.medications);
        applyAutoField('vETA', f.eta);
        updateMapDisplay(); updateAutoNote(); markStale();
      }
    } finally {
      aiExtractInFlight = false;
      if (btn) { btn.disabled = false; btn.textContent = origTxt; }
    }
  }
  $('aiExtractBtn').addEventListener('click', () => runSmartAiExtraction($('complaintText').value));

  function resetAllFieldStatuses() {
    ALL_EDITABLE_FIELDS.forEach(id => { autoFlag[id] = false; manualFlag[id] = false; const el = $(id); if (el) el.classList.remove('auto-filled', 'manually-edited'); });
    refreshManualBadges(); updateAutoNote(); updateMapDisplay();
  }

  const micBtn = $('micBtn'), waveWrap = $('waveWrap'), canvas = $('waveCanvas'), micStatus = $('micStatus'), micError = $('micError');
  let stream = null, audioCtx = null, analyser = null, raf = null, recTimer = null, recStart = 0, recording = false, lastLoud = 0;
  const showError = msg => { micError.style.display = 'block'; micError.textContent = '⚠️ ' + msg; };
  const hideError = () => { micError.style.display = 'none'; };
  const setStatus = (msg, color) => { micStatus.textContent = msg; micStatus.style.color = color || '#9ca3af'; };

  function drawWave() {
    const ctx = canvas.getContext('2d'), W = canvas.width = canvas.clientWidth || 600, H = canvas.height = 64;
    const data = new Uint8Array(analyser.frequencyBinCount), bars = 48, bw = W / bars;
    (function frame() {
      analyser.getByteFrequencyData(data); ctx.clearRect(0, 0, W, H);
      let sum = 0; for (let i = 0; i < data.length; i++) sum += data[i];
      if (sum / data.length > 8) lastLoud = Date.now();
      for (let i = 0; i < bars; i++) {
        const v = data[Math.floor(i * data.length * 0.6 / bars)] / 255, h = Math.max(4, v * H);
        ctx.fillStyle = '#38bdf8'; ctx.fillRect(i * bw + 2, (H - h) / 2, bw - 4, h);
      }
      raf = requestAnimationFrame(frame);
    })();
  }

  const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  let live = null, liveOn = false, liveMode = false, liveBase = '', liveSession = '', liveStopTimer = null, liveRestarts = 0;
  const composeLive = interim => [liveBase, liveSession, interim].filter(x => x && x.trim()).join(' ').replace(/\s{2,}/g, ' ').trim();

  function beginRecognition() {
    const r = new SR(); live = r; r.lang = $('micLang').value || 'ar-EG'; r.continuous = true; r.interimResults = true;
    r.onresult = ev => {
      let fin = '', interim = '';
      for (let i = 0; i < ev.results.length; i++) {
        const res = ev.results[i], tx = res[0].transcript;
        if (res.isFinal) fin += tx + ' '; else interim += tx + ' ';
      }
      liveSession = fin.trim();
      const combined = composeLive(interim.trim());
      if (!manualFlag.complaintText) { $('complaintText').value = combined; autoFlag.complaintText = true; $('complaintText').classList.add('auto-filled'); }
      markStale();
    };
    r.onerror = ev => { if (ev.error !== 'no-speech' && ev.error !== 'aborted') { liveOn = false; showError('خطأ في التعرف على الكلام: ' + ev.error); setTimeout(finishLive, 50); } };
    r.onend = () => {
      if (live !== r) return;
      if (liveOn && liveRestarts < 200) { liveBase = composeLive(''); liveSession = ''; liveRestarts++; setTimeout(() => { if (liveOn) beginRecognition(); }, 100); }
      else finishLive();
    };
    try { r.start(); } catch (e) { liveOn = false; finishLive(); }
  }

  async function startLive() {
    hideError(); liveOn = true; liveMode = true; recording = true; liveRestarts = 0;
    liveBase = $('complaintText').value.trim(); liveSession = '';
    micBtn.classList.add('rec'); micBtn.textContent = '⏹ إيقاف'; waveWrap.style.display = 'block';
    recStart = Date.now(); lastLoud = 0; setStatus('🎧 جاري الاستماع… تحدث');
    recTimer = setInterval(() => {
      const sec = Math.floor((Date.now() - recStart) / 1000);
      const tt = String(Math.floor(sec / 60)).padStart(2, '0') + ':' + String(sec % 60).padStart(2, '0');
      setStatus((Date.now() - lastLoud < 1200 ? '🎙 المايك شغال ✓ — ' : '🎧 جاري الاستماع… — ') + tt, Date.now() - lastLoud < 1200 ? '#34d399' : '#fbbf24');
    }, 300);
    beginRecognition();
    try {
      stream = await navigator.mediaDevices.getUserMedia({ audio: { echoCancellation: true, noiseSuppression: true } });
      audioCtx = new (window.AudioContext || window.webkitAudioContext)();
      analyser = audioCtx.createAnalyser(); analyser.fftSize = 128;
      audioCtx.createMediaStreamSource(stream).connect(analyser);
      if (recording) drawWave();
    } catch (e) {}
  }

  function stopLive() { liveOn = false; try { if (live) live.stop(); } catch (e) {} clearTimeout(liveStopTimer); liveStopTimer = setTimeout(finishLive, 800); }

  async function finishLive() {
    clearTimeout(liveStopTimer);
    if (!recording || !liveMode) return;
    recording = false; liveMode = false; liveOn = false; clearInterval(recTimer); cancelAnimationFrame(raf);
    waveWrap.style.display = 'none';
    if (stream) stream.getTracks().forEach(t => t.stop());
    if (audioCtx) { try { audioCtx.close(); } catch (e) {} }
    micBtn.classList.remove('rec'); micBtn.textContent = '🎤 ابدأ التسجيل';
    const txt = composeLive('');
    if (txt) {
      if (!manualFlag.complaintText) $('complaintText').value = txt;
      $('asrNote').style.display = 'block';
      markStale();
      await runSmartAiExtraction(txt);
    }
  }

  micBtn.addEventListener('click', () => {
    if (recording) { stopLive(); }
    else if (SR) { startLive(); }
    else { showError('يرجى استخدام متصفح Chrome أو Edge للتعرف الصوتي المباشر.'); }
  });

  let activeParamedic = null, pendingDraft = null;
  $('loginBtn').addEventListener('click', async () => {
    const code = $('loginCode').value.trim().toUpperCase(), pin = $('loginPin').value.trim(), status = $('loginStatus');
    if (!code || !pin) { status.textContent = 'أدخل كود المسعف والرقم السري.'; return; }
    $('loginBtn').disabled = true; status.textContent = 'جارٍ التحقق…';
    try {
      const auth = await callPy('notebook.authenticate_paramedic_v1', [code, pin], 10000);
      if (!auth || !auth.ok) { status.textContent = 'كود المسعف أو الرقم السري غير صحيح.'; return; }
      activeParamedic = { code: auth.code, name: auth.name };
      $('reviewReviewer').value = activeParamedic.code;
      $('activeParamedicLabel').textContent = 'المسعف المسجل: ' + activeParamedic.name + ' (' + activeParamedic.code + ')';
      $('loginScreen').style.display = 'none'; $('appWorkspace').style.display = 'block';
      $('loginPin').value = ''; status.textContent = '';
    } catch (e) { status.textContent = 'تعذر تسجيل الدخول: ' + (e.message || e); }
    finally { $('loginBtn').disabled = false; }
  });
  $('loginPin').addEventListener('keydown', e => { if (e.key === 'Enter') $('loginBtn').click(); });
  $('logoutBtn').addEventListener('click', () => { activeParamedic = null; $('appWorkspace').style.display = 'none'; $('loginScreen').style.display = 'block'; });

  $('clearCaseBtn').addEventListener('click', () => $('resetModalBackdrop').classList.add('open'));
  $('resetCancelBtn').addEventListener('click', () => $('resetModalBackdrop').classList.remove('open'));
  $('resetConfirmBtn').addEventListener('click', async () => {
    $('resetModalBackdrop').classList.remove('open');
    if (recording) stopLive();
    liveBase = ''; liveSession = '';
    ALL_EDITABLE_FIELDS.forEach(id => { if ($(id)) $(id).value = ''; });
    resetAllFieldStatuses();
    const oldId = pendingDraft ? pendingDraft.id : '';
    closeReview(); $('resultCard').style.display = 'none'; $('savedBanner').style.display = 'none'; $('asrNote').style.display = 'none'; clearAppErr();
    if (oldId) { try { await callPy('notebook.clear_case_drafts_v1', [oldId], 5000); } catch (e) {} }
  });

  $('emergencyBtn').addEventListener('click', async () => {
    if (!window.confirm('هل تريد إرسال تنبيه طوارئ أحمر؟')) return;
    const st = $('emergencyStatus'); $('emergencyBtn').disabled = true;
    try {
      const r = await callPy('notebook.emergency_alert_v1', [activeParamedic ? activeParamedic.code : ''], 12000);
      st.textContent = r && r.message ? r.message : '';
    } finally { $('emergencyBtn').disabled = false; }
  });

  document.querySelectorAll('.eta-chip').forEach(btn => btn.addEventListener('click', () => { $('vETA').value = btn.getAttribute('data-eta'); markFieldManuallyEdited('vETA'); markStale(); }));

  const STAT = { RED: ['#7f1d1d', '🔴'], ORANGE: ['#9a3412', '🟠'], WATCH: ['#854d0e', '🟡'], OK: ['#14532d', '🟢'], NA: ['#374151', '⚪'] };
  function vitalsHtml(v) {
    if (!v) return "";
    const chip = (title, text, st) => `<div style='background:${STAT[st][0]};padding:6px 10px;border-radius:6px;flex:1;min-width:140px;'><div style='font-size:11px;opacity:.85;'>${esc(title)}</div><b>${STAT[st][1]} ${esc(text)}</b></div>`;
    const bp = (v.sbp != null || v.dbp != null) ? `${v.sbp ?? '؟'}/${v.dbp ?? '؟'} mmHg` : 'لم يُقَس';
    const sp = [v.spo2 != null ? 'SpO₂ ' + v.spo2 + '%' : '', v.rr != null ? 'RR ' + v.rr : ''].filter(Boolean).join(' | ') || 'لم يُقَس';
    const circ = [bp !== 'لم يُقَس' ? bp : '', v.map != null ? 'MAP ' + v.map : '', v.hr != null ? 'HR ' + v.hr : ''].filter(Boolean).join(' | ') || 'لم يُقَس';
    const cons = [v.avpu ? 'AVPU ' + v.avpu : '', v.gcs != null ? 'GCS ' + v.gcs + '/15' : ''].filter(Boolean).join(' | ') || 'لم يُقيّم';
    let h = `<div style='margin-top:10px;display:flex;gap:8px;flex-wrap:wrap;'>${chip('B — التنفس', sp, v.b || 'NA')}${chip('C — الدورة و MAP', circ, v.c || 'NA')}${chip('D — الوعي', cons, v.d || 'NA')}${v.temp != null ? chip('E — الحرارة', v.temp + '°C', v.e || 'NA') : ''}</div>`;
    if (v.reasons && v.reasons.length) h += "<ul style='margin:8px 0 0;padding-inline-start:18px;font-weight:bold;'>" + v.reasons.map(r => "<li>" + esc(r) + "</li>").join('') + "</ul>";
    return h;
  }

  (function buildReview() {
    $('reviewDepartment').innerHTML = CATALOG.departments.concat([CATALOG.dept_other]).map(d => `<option value='${esc(d)}'>${esc(d)}</option>`).join('');
    $('reviewPrep').innerHTML = CATALOG.prep.map(g =>
      `<fieldset style='border:1px solid #e8d9a8;border-radius:8px;padding:6px 10px 8px;margin:0 0 8px;'><legend style='font-weight:bold;font-size:13px;padding:0 6px;'>${esc(g.title)}</legend><div style='display:flex;flex-wrap:wrap;gap:4px 14px;'>` +
      g.items.map(it => `<label style='font-size:13px;display:flex;gap:4px;align-items:center;cursor:pointer;'><input type='checkbox' data-prep='${esc(g.key)}' value='${esc(it)}'> ${esc(it)}</label>`).join('') + "</div></fieldset>"
    ).join('');
  })();

  const prepBoxes = () => Array.from(document.querySelectorAll('#reviewPrep input[type=checkbox]'));
  function readPrep() { const o = {}; CATALOG.prep.forEach(g => o[g.key] = []); prepBoxes().forEach(c => { if (c.checked) o[c.getAttribute('data-prep')].push(c.value); }); return o; }
  function setPrep(d) { prepBoxes().forEach(c => c.checked = (d[c.getAttribute('data-prep')] || []).indexOf(c.value) !== -1); }
  function renderChips() {
    const o = readPrep(), rows = CATALOG.prep.filter(g => o[g.key].length).map(g => `<div>${esc(g.title + ': ' + o[g.key].join('، '))}</div>`);
    if ($('reviewOtherNote').value.trim()) rows.push(`<div>📌 ${esc($('reviewOtherNote').value.trim())}</div>`);
    $('reviewChips').innerHTML = rows.length ? rows.join('') : "<div style='color:#888;'>لا توجد تجهيزات مقترحة.</div>";
  }
  $('reviewPrep').addEventListener('change', renderChips);
  $('reviewOtherNote').addEventListener('input', renderChips);

  const TIER_RANK = { GREEN: 0, ORANGE: 1, RED: 2 };
  function updateOverrideVisibility() {
    const floor = pendingDraft && pendingDraft.floor, need = !!floor && TIER_RANK[$('reviewTier').value] < TIER_RANK[floor];
    $('reviewOverrideWrap').style.display = need ? 'block' : 'none';
    if (!need) $('reviewOverride').value = '';
  }
  $('reviewTier').addEventListener('change', updateOverrideVisibility);
  $('reviewDepartment').addEventListener('change', () => $('reviewDeptOther').style.display = $('reviewDepartment').value === CATALOG.dept_other ? 'block' : 'none');

  function markStale() { if (pendingDraft) { $('reviewStale').style.display = 'block'; $('confirmSaveBtn').disabled = true; } }
  function closeReview() { pendingDraft = null; updateOverrideVisibility(); $('reviewPanel').style.display = 'none'; $('reviewStale').style.display = 'none'; $('reviewConfirmed').checked = false; $('confirmSaveBtn').disabled = false; $('saveStatus').textContent = ''; }
  function showBanner(text, kind) { const b = $('savedBanner'); b.style.display = 'block'; b.style.background = kind === 'ok' ? '#e8f5e9' : '#fff4d6'; b.style.color = kind === 'ok' ? '#1b5e20' : '#654d03'; b.style.border = '1px solid ' + (kind === 'ok' ? '#a5d6a7' : '#e8bd57'); b.textContent = text; }

  const colorMap = { "RED": ["#d9534f", "🔴 اقتراح آلي: فئة حمراء (إنعاش فوري)"], "ORANGE": ["#e67e22", "🟠 اقتراح آلي: فئة برتقالية (عاجلة)"], "GREEN": ["#28a745", "🟢 اقتراح آلي: فئة خضراء (روتينية)"] };

  $('submitBtn').addEventListener('click', async function () {
    if (!activeParamedic) { appErr('سجّل دخول المسعف أولاً.'); return; }
    const pName = $('pName').value.trim() || "غير مسجل", pAge = $('pAge').value.trim() || "-", pGender = $('pGender').value, pNationalId = nid.value.trim() || "-";
    let userText = $('complaintText').value.trim();
    const cLbl = $('vConsciousness').value ? ('AVPU=' + $('vConsciousness').value) : '', gcsV = $('vGCS').value.trim();
    if (cLbl || gcsV) userText += '\n[Consciousness: ' + [cLbl, gcsV ? ('GCS=' + gcsV) : ''].filter(Boolean).join(', ') + ']';
    clearAppErr();
    if (!userText) { appErr("يرجى إدخال شكوى المريض أولاً."); return; }
    if (pNationalId !== "-" && !/^[0-9]{14}$/.test(pNationalId)) { appErr("الرقم القومي يجب أن يتكون من 14 رقماً."); return; }

    const btn = $('submitBtn'), loading = $('triageLoading');
    btn.disabled = true; $('resultCard').style.display = "none"; $('savedBanner').style.display = 'none'; closeReview();
    loading.style.display = "block"; loading.textContent = "⏳ جاري إنشاء المسودة الآلية...";
    try {
      const tempVal = $('vTemp').value.trim();
      const tempC = tempVal ? (uiLanguage === 'en' ? String(Math.round(((Number(tempVal) - 32) * 5 / 9) * 10) / 10) : tempVal) : '';
      const d = await callPy('notebook.triage_clinical_exact_v3', [pName, pAge, pNationalId, userText, $('vSys').value.trim(), $('vDia').value.trim(), $('vSpo2').value.trim(), $('vRR').value.trim(), tempC, $('vMeds').value.trim(), uiLanguage, $('vHR').value.trim(), $('vETA').value.trim(), pGender]);
      if (!d || d.error) { appErr("تعذر إنشاء المسودة: " + ((d && d.error) || '')); return; }

      const info = colorMap[(d.tier || "").toUpperCase()] || colorMap["ORANGE"];
      const card = $('resultCard'); card.style.backgroundColor = info[0]; card.style.display = "block";
      card.innerHTML = `<div style='font-weight:bold;font-size:15px;margin-bottom:8px;'>📝 مسودة اقتراح آلي — ليست قرار فرز معتمداً</div><div style='display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid rgba(255,255,255,.4);padding-bottom:8px;margin-bottom:10px;'><h3 style='margin:0;color:white;'>${info[1]}</h3><span style='background:rgba(0,0,0,.3);padding:5px 12px;border-radius:16px;font-weight:bold;'>درجة مقترحة: ${esc(d.risk_score)} / 10</span></div><p style='margin:5px 0;'><b>🆔 رقم المريض:</b> <span style='background:rgba(0,0,0,.28);padding:2px 8px;border-radius:6px;font-family:monospace;'>${esc(d.patient_id)}</span> | <b>المريض:</b> ${esc(pName)} | <b>العمر:</b> ${esc(pAge)} | <b>النوع:</b> ${esc(d.gender)}</p>${vitalsHtml(d.vitals)}<p style='margin:8px 0 5px;'><b>الملخص المقترح:</b> ${esc(d.clinical_summary)}</p><p style='margin:5px 0;'><b>القسم المقترح:</b> ${esc(d.target_department)}</p>`;

      $('reviewModelNote').innerHTML = `🤖 النموذج: <b>${esc(d.model)}</b>`;
      $('reviewSnapshot').innerHTML = `<b>🆔 رقم المريض المولَّد:</b> <span style='color:#0284c7;font-family:monospace;font-weight:bold;'>${esc(d.patient_id)}</span> | <b>المريض:</b> ${esc(pName)} | ${esc(pAge)} سنة | <b>النوع:</b> ${esc(d.gender)}<br><b>الشكوى الحرفية:</b> ${esc(userText)}`;
      $('reviewSummary').value = d.clinical_summary || '';
      $('reviewTier').value = (d.tier || 'ORANGE').toUpperCase();
      $('reviewScore').value = d.risk_score;
      $('reviewDepartment').value = d.target_department || CATALOG.departments[0];
      setPrep(d); $('reviewOtherNote').value = d.other_note || '';
      $('reviewReviewer').value = activeParamedic.code;
      renderChips();
      pendingDraft = { id: d.draft_id, floor: d.vitals_floor || null };
      updateOverrideVisibility();
      $('reviewPanel').style.display = 'block';
    } catch (err) {
      appErr("حدث خطأ: " + ((err && err.message) || err));
    } finally { loading.style.display = "none"; btn.disabled = false; }
  });

  $('confirmSaveBtn').addEventListener('click', async function () {
    const st = $('saveStatus'); st.style.color = '#b71c1c'; st.textContent = '';
    if (!pendingDraft) { st.textContent = '⚠️ لا توجد مسودة جاهزة للحفظ.'; return; }
    if (!$('reviewConfirmed').checked) { st.textContent = '⚠️ يجب وضع علامة صح على مربع التأكيد أعلاه أولاً.'; return; }
    const score = Number($('reviewScore').value), tier = $('reviewTier').value;
    if ((tier === 'RED' && score < 9) || (tier === 'ORANGE' && (score < 5 || score > 8)) || (tier === 'GREEN' && score > 4)) {
      st.textContent = '⚠️ الفئة ودرجة الخطورة غير متسقتين.'; return;
    }
    const payload = {
      draft_id: pendingDraft.id, confirmed: true, reviewer: activeParamedic.code,
      tier: tier, score: score, summary: $('reviewSummary').value.trim(),
      department: $('reviewDepartment').value, department_other: $('reviewDeptOther').value.trim(),
      prep: readPrep(), other_note: $('reviewOtherNote').value.trim(), override_reason: $('reviewOverride').value.trim()
    };
    const origText = this.textContent; this.disabled = true; this.textContent = '⏳ جارٍ الحفظ والإرسال للمستشفى...';
    try {
      const d = await callPy('notebook.confirm_and_save_v1', [JSON.stringify(payload)], 20000);
      if (!d || d.error) { st.textContent = '❌ تعذر الحفظ: ' + ((d && d.error) || 'لم يصل رد'); return; }
      $('patientsListBody').innerHTML = d.table_html;
      showBanner('✅ تم الحفظ (رقم المريض: ' + d.patient_id + ') وإرسال الحالة لشاشة الطوارئ بالمستشفى | ' + d.notify, 'ok');
      closeReview(); $('resultCard').style.display = 'none';
      ALL_EDITABLE_FIELDS.forEach(id => { if ($(id)) $(id).value = ''; });
      resetAllFieldStatuses(); $('asrNote').style.display = 'none';
    } catch (err) {
      st.textContent = '❌ خطأ في الاتصال: ' + ((err && err.message) || err);
    } finally { this.disabled = false; this.textContent = origText; }
  });

  $('cancelReviewBtn').addEventListener('click', () => { closeReview(); $('resultCard').style.display = 'none'; });
  setUiLanguage(uiLanguage); updateMapDisplay();
})();
</script>
"""

# ============================================================
# 6b. شاشة استقبال الطوارئ بالمستشفى — RELAY ER Live Dashboard
# ============================================================
ER_DASHBOARD_HTML = r"""<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>RELAY — Bridging the Gap Before Arrival | ER Live Dashboard</title>
<style>
  :root { --navy: #0a192f; --navy-mid: #0f2942; --blue: #0284c7; --cyan: #38bdf8; --bg: #f1f5f9; --surface: #ffffff; --ink: #0f172a; --muted: #64748b; --line: #dbe4ef; }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--ink); font-family: system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif; }
  .er-header { background: linear-gradient(135deg, #0a192f 0%, #0f2942 65%, #0369a1 100%); color: #fff; padding: 14px 24px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 12px; box-shadow: 0 4px 14px rgba(10,25,47,.2); }
  .er-brand { display: flex; align-items: center; gap: 12px; }
  .er-logo { width: 46px; height: 46px; border-radius: 12px; background: rgba(56,189,248,.16); border: 1px solid rgba(56,189,248,.45); display: grid; place-items: center; font-size: 24px; }
  .er-title { font-size: 20px; font-weight: 900; margin: 0; letter-spacing: 0.4px; }
  .er-sub { font-size: 12px; color: #7dd3fc; margin-top: 2px; }
  .er-controls { display: flex; align-items: center; gap: 10px; flex-wrap: wrap; }
  .live-pill { display: inline-flex; align-items: center; gap: 7px; background: rgba(16,185,129,.18); border: 1px solid #10b981; color: #a7f3d0; padding: 6px 12px; border-radius: 999px; font-size: 12.5px; font-weight: 700; }
  .live-dot { width: 9px; height: 9px; border-radius: 50%; background: #10b981; animation: blink 1.4s infinite; }
  @keyframes blink { 0%, 100% { opacity: 1; } 50% { opacity: 0.25; } }
  .sound-btn { background: rgba(255,255,255,.14); color: #fff; border: 1px solid rgba(255,255,255,.3); padding: 7px 13px; border-radius: 9px; font-weight: 700; font-size: 12.5px; cursor: pointer; font-family: inherit; }
  .container { max-width: 1220px; margin: 18px auto; padding: 0 16px 32px; }
  .kpi-grid { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 14px; margin-bottom: 18px; }
  .kpi-card { background: var(--surface); border: 1px solid var(--line); border-radius: 14px; padding: 14px 18px; border-inline-start: 5px solid var(--blue); box-shadow: 0 2px 6px rgba(15,41,66,.04); }
  .kpi-card.red { border-inline-start-color: #d9534f; } .kpi-card.orange { border-inline-start-color: #e67e22; } .kpi-card.green { border-inline-start-color: #28a745; }
  .kpi-label { font-size: 12.5px; font-weight: 700; color: var(--muted); }
  .kpi-val { font-size: 26px; font-weight: 900; color: var(--navy-mid); margin-top: 4px; }
  .cases-stack { display: flex; flex-direction: column; gap: 16px; }
  .case-card { background: var(--surface); border: 1px solid var(--line); border-radius: 16px; overflow: hidden; box-shadow: 0 4px 14px rgba(15,41,66,.06); }
  .case-card.new-flash { animation: cardHighlight 1.8s ease-out; }
  @keyframes cardHighlight { 0% { box-shadow: 0 0 0 6px rgba(217,83,79,.45); } 100% { box-shadow: 0 4px 14px rgba(15,41,66,.06); } }
  .case-banner { padding: 12px 18px; color: #fff; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px; }
  .case-banner.RED { background: linear-gradient(135deg, #b91c1c, #dc2626); }
  .case-banner.ORANGE { background: linear-gradient(135deg, #c2410c, #ea580c); }
  .case-banner.GREEN { background: linear-gradient(135deg, #15803d, #16a34a); }
  .case-badge { background: rgba(0,0,0,.28); padding: 4px 11px; border-radius: 999px; font-weight: 800; font-size: 13px; }
  .eta-pill { background: #fff; color: #0f172a; padding: 5px 12px; border-radius: 999px; font-weight: 850; font-size: 13px; }
  .case-body { padding: 16px 18px; }
  .patient-strip { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px; padding-bottom: 12px; border-bottom: 1px solid #eef2f6; margin-bottom: 12px; }
  .dept-tag { background: #e0f2fe; color: #0369a1; border: 1px solid #bae6fd; padding: 6px 12px; border-radius: 9px; font-weight: 800; font-size: 13.5px; }
  .vitals-bar { display: grid; grid-template-columns: repeat(auto-fit, minmax(120px, 1fr)); gap: 10px; margin-bottom: 14px; }
  .v-box { background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 10px; padding: 8px 12px; }
  .v-lbl { font-size: 11px; font-weight: 700; color: var(--muted); }
  .v-num { font-size: 15.5px; font-weight: 850; color: var(--navy-mid); margin-top: 2px; }
  .summary-box { background: #f8fafc; border-inline-start: 4px solid var(--navy-mid); padding: 10px 14px; border-radius: 8px; font-size: 14px; line-height: 1.6; margin-bottom: 12px; }
  .meds-alert { background: #fff1f2; border: 1px solid #fecdd3; color: #9f1239; padding: 9px 14px; border-radius: 9px; font-size: 13.5px; font-weight: 750; margin-bottom: 12px; }
  .prep-grid { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 8px; }
  .prep-tag { background: #f1f5f9; border: 1px solid #cbd5e1; color: #1e293b; padding: 5px 10px; border-radius: 8px; font-size: 12.5px; font-weight: 650; }
  .case-footer { margin-top: 12px; padding-top: 10px; border-top: 1px solid #eef2f6; display: flex; justify-content: space-between; flex-wrap: wrap; gap: 8px; font-size: 11.5px; color: var(--muted); }
  .empty-state { background: var(--surface); border: 1px dashed #cbd5e1; border-radius: 16px; padding: 48px 20px; text-align: center; color: var(--muted); }
  @media(max-width: 720px) { .kpi-grid { grid-template-columns: 1fr 1fr; } }
</style>
</head>
<body>
  <header class="er-header">
    <div class="er-brand">
      <div class="er-logo">🏥</div>
      <div>
        <h1 class="er-title">RELAY — لوحة استقبال طوارئ المستشفى</h1>
        <div class="er-sub">Bridging the Gap Before Arrival &nbsp;|&nbsp; استقبال بلاغات الإسعاف لحظياً وتجهيز غرف الإنعاش</div>
      </div>
    </div>
    <div class="er-controls">
      <div class="live-pill"><span class="live-dot"></span> <span id="syncStatus">متصل لحظياً</span></div>
      <button type="button" id="soundToggleBtn" class="sound-btn">🔔 التنبيه الصوتي: مفعّل</button>
    </div>
  </header>
  <main class="container">
    <section class="kpi-grid">
      <div class="kpi-card"><div class="kpi-label">إجمالي الحالات</div><div class="kpi-val" id="kpiTotal">0</div></div>
      <div class="kpi-card red"><div class="kpi-label">🔴 حالات حمراء (إنعاش فوري)</div><div class="kpi-val" id="kpiRed" style="color:#b91c1c;">0</div></div>
      <div class="kpi-card orange"><div class="kpi-label">🟠 حالات برتقالية (عاجلة)</div><div class="kpi-val" id="kpiOrange" style="color:#c2410c;">0</div></div>
      <div class="kpi-card green"><div class="kpi-label">🟢 حالات خضراء (مستقرة)</div><div class="kpi-val" id="kpiGreen" style="color:#15803d;">0</div></div>
    </section>
    <section id="casesContainer" class="cases-stack"><div class="empty-state">⏳ جاري تحميل بلاغات الإسعاف الواردة...</div></section>
  </main>
<script>
(function () {
  const CALL_TOKEN = "__CALL_TOKEN__";
  const esc = s => String(s == null ? "" : s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
  let soundEnabled = true, knownCount = null;
  document.getElementById('soundToggleBtn').addEventListener('click', function () {
    soundEnabled = !soundEnabled;
    this.textContent = soundEnabled ? '🔔 التنبيه الصوتي: مفعّل' : '🔕 التنبيه الصوتي: صامت';
    if (soundEnabled) playAlertChime('ORANGE');
  });
  function playAlertChime(tier) {
    if (!soundEnabled) return;
    try {
      const ctx = new (window.AudioContext || window.webkitAudioContext)();
      (tier === 'RED' ? [880, 1174.66, 880, 1174.66] : [659.25, 880]).forEach((freq, idx) => {
        const osc = ctx.createOscillator(), gain = ctx.createGain();
        osc.frequency.value = freq;
        gain.gain.setValueAtTime(0.18, ctx.currentTime + idx * 0.16);
        gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + idx * 0.16 + 0.15);
        osc.connect(gain); gain.connect(ctx.destination);
        osc.start(ctx.currentTime + idx * 0.16); osc.stop(ctx.currentTime + idx * 0.16 + 0.15);
      });
    } catch (e) {}
  }
  function renderCases(cases, highlightNewest) {
    document.getElementById('kpiTotal').textContent = cases.length;
    document.getElementById('kpiRed').textContent = cases.filter(c => c.tier === 'RED').length;
    document.getElementById('kpiOrange').textContent = cases.filter(c => c.tier === 'ORANGE').length;
    document.getElementById('kpiGreen').textContent = cases.filter(c => c.tier === 'GREEN').length;
    const container = document.getElementById('casesContainer');
    if (!cases.length) {
      container.innerHTML = '<div class="empty-state"><h3>🚑 لا توجد حالات واردة حالياً</h3><p>ستظهر هنا أي حالة يؤكدها المسعف في سيارة الإسعاف فوراً.</p></div>';
      return;
    }
    container.innerHTML = cases.map((c, idx) => {
      const icon = c.tier === 'RED' ? '🔴' : (c.tier === 'ORANGE' ? '🟠' : '🟢');
      const title = c.tier === 'RED' ? 'حالة حمراء — إنعاش فوري' : (c.tier === 'ORANGE' ? 'حالة برتقالية — عاجلة' : 'حالة خضراء — روتينية');
      const etaHtml = (c.eta && c.eta !== '-') ? `<span class="eta-pill">⏱️ الوصول المتوقع: ${esc(c.eta)} دقيقة</span>` : `<span class="eta-pill">⏱️ ETA غير محدد</span>`;
      const medsHtml = (c.meds && c.meds !== '-') ? `<div class="meds-alert">💊 الأدوية الحالية (تنبيه التداخلات الدوائية): ${esc(c.meds)}</div>` : '';
      const prepHtml = (c.prep_sections && c.prep_sections.length) ? `<div style="margin-top:8px;"><div style="font-size:12px;font-weight:800;color:#334155;margin-bottom:4px;">📋 التجهيزات المطلوبة قبل الوصول:</div><div class="prep-grid">${c.prep_sections.map(p => `<span class="prep-tag">${esc(p)}</span>`).join('')}</div></div>` : '';
      return `<article class="case-card${(highlightNewest && idx === 0) ? ' new-flash' : ''}"><div class="case-banner ${esc(c.tier)}"><div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap;"><span class="case-badge">#${esc(c.case_no)}</span><span class="case-badge" style="font-family:monospace;letter-spacing:0.4px;">🆔 ${esc(c.patient_id)}</span><strong style="font-size:16px;">${icon} ${title}</strong><span class="case-badge">الخطورة: ${esc(c.score)} / 10</span></div><div>${etaHtml}</div></div><div class="case-body"><div class="patient-strip"><div style="font-size:14.5px;"><b>👤 المريض:</b> ${esc(c.name)} &nbsp;|&nbsp; <b>العمر:</b> ${esc(c.age)} سنة &nbsp;|&nbsp; <b>النوع:</b> ${esc(c.gender)} &nbsp;|&nbsp; <b>الرقم القومي:</b> ${esc(c.national_id)}</div><div class="dept-tag">🏥 القسم المستهدف: ${esc(c.department)}</div></div><div class="vitals-bar"><div class="v-box"><div class="v-lbl">🩸 ضغط الدم</div><div class="v-num">${esc(c.bp)} mmHg</div></div><div class="v-box"><div class="v-lbl">📊 متوسط الضغط MAP</div><div class="v-num">${esc(c.map)} mmHg</div></div><div class="v-box"><div class="v-lbl">💓 النبض HR</div><div class="v-num">${esc(c.hr)} bpm</div></div><div class="v-box"><div class="v-lbl">🫁 الأكسجين SpO₂</div><div class="v-num">${esc(c.spo2)}%</div></div><div class="v-box"><div class="v-lbl">🌬️ التنفس RR</div><div class="v-num">${esc(c.rr)} /د</div></div><div class="v-box"><div class="v-lbl">🌡️ الحرارة</div><div class="v-num">${esc(c.temp)} °C</div></div><div class="v-box"><div class="v-lbl">🧠 الوعي AVPU/GCS</div><div class="v-num">${esc(c.avpu)} | ${esc(c.gcs)}</div></div></div><div class="summary-box"><b>📝 الملخص السريري:</b> ${esc(c.complaint)}</div>${medsHtml}${prepHtml}<div class="case-footer"><span>🕒 وقت البلاغ: ${esc(c.timestamp)} | 👨‍🚑 المسعف: <b>${esc(c.reviewer)}</b></span><span>🤖 النموذج: ${esc(c.model)} (${esc(c.edited)})</span></div></div></article>`;
    }).join('');
  }
  async function pollCases() {
    try {
      const r = await fetch('call', { method: 'POST', headers: { 'Content-Type': 'application/json', 'X-Token': CALL_TOKEN }, body: JSON.stringify({ name: 'notebook.get_er_cases_v1', args: [] }) });
      if (!r.ok) return;
      const d = await r.json(), cases = (d && d.cases) || [];
      const isNew = (knownCount !== null && cases.length > knownCount);
      if (isNew && cases.length > 0) playAlertChime(cases[0].tier);
      knownCount = cases.length;
      renderCases(cases, isNew);
      document.getElementById('syncStatus').textContent = 'متصل لحظياً — آخر تحديث: ' + new Date().toLocaleTimeString('ar-EG');
    } catch (e) {}
  }
  pollCases(); setInterval(pollCases, 3000);
})();
</script>
</body>
</html>
"""

# ============================================================
# 7. ربط الدوال بالخادم (Colab Kernel + Paramedic App + ER Dashboard)
# ============================================================
HANDLERS = {
    "notebook.status_v1": status_v1,
    "notebook.authenticate_paramedic_v1": authenticate_paramedic_v1,
    "notebook.emergency_alert_v1": emergency_alert_v1,
    "notebook.triage_clinical_exact_v3": triage_clinical_exact_v3,
    "notebook.confirm_and_save_v1": confirm_and_save_v1,
    "notebook.transcribe_audio_v1": transcribe_audio_v1,
    "notebook.extract_fields_v1": extract_fields_v1,
    "notebook.clear_case_drafts_v1": clear_case_drafts_v1,
    "notebook.get_er_cases_v1": get_er_cases_v1,
}

for _name, _fn in HANDLERS.items():
    output.register_callback(_name, _fn)

threading.Thread(target=_telegram_startup_worker, daemon=True).start()

_CATALOG_JSON = json.dumps({
    "departments": DEPT_CATALOG,
    "dept_other": DEPT_OTHER,
    "prep": [{"key": k, "title": PREP_UI[k][0], "note": PREP_UI[k][1], "items": READINESS_CATALOG[k]} for k, _ in PREP_LABELS],
}, ensure_ascii=False)

_PROXY_BASE_URL = ""

def _render_html():
    er_href = f"{_PROXY_BASE_URL}/er?k={APP_TOKEN}" if _PROXY_BASE_URL else f"/er?k={APP_TOKEN}"
    return (APP_HTML
            .replace("__INITIAL_ROWS__", get_saved_rows_html())
            .replace("__CATALOG_JSON__", _CATALOG_JSON)
            .replace("__CALL_TOKEN__", APP_TOKEN)
            .replace("__ER_URL__", er_href))

def _render_er_html():
    return ER_DASHBOARD_HTML.replace("__CALL_TOKEN__", APP_TOKEN)

class _StandaloneHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass
    def do_GET(self):
        parsed = urlparse(self.path)
        qs = parse_qs(parsed.query)
        if qs.get("k", [""])[0] != APP_TOKEN:
            self.send_response(403); self.end_headers(); self.wfile.write(b"Forbidden"); return
        page = _render_er_html().encode("utf-8") if parsed.path.rstrip("/") == "/er" else f"<!DOCTYPE html><html><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'></head><body style='margin:0;background:#f1f5f9;'>{_render_html()}</body></html>".encode("utf-8")
        self.send_response(200); self.send_header("Content-Type", "text/html; charset=utf-8"); self.end_headers(); self.wfile.write(page)

    def do_POST(self):
        try:
            if self.headers.get("X-Token") != APP_TOKEN:
                self.send_response(403); self.end_headers(); return
            length = int(self.headers.get("Content-Length", 0))
            body = json.loads(self.rfile.read(length).decode("utf-8"))
            fn = HANDLERS.get(body.get("name"))
            if not fn:
                self.send_response(404); self.end_headers(); return
            res = fn(*(body.get("args") or []))
            raw_data = res.data if hasattr(res, "data") else res
            out_bytes = raw_data.encode("utf-8") if isinstance(raw_data, str) else json.dumps(raw_data, ensure_ascii=False).encode("utf-8")
            self.send_response(200); self.send_header("Content-Type", "application/json; charset=utf-8"); self.end_headers(); self.wfile.write(out_bytes)
        except Exception as ex:
            err_bytes = json.dumps({"error": f"{type(ex).__name__}: {ex}"}, ensure_ascii=False).encode("utf-8")
            self.send_response(200); self.send_header("Content-Type", "application/json; charset=utf-8"); self.end_headers(); self.wfile.write(err_bytes)

_srv = ThreadingHTTPServer(("127.0.0.1", 0), _StandaloneHandler)
_port = _srv.server_address[1]
threading.Thread(target=_srv.serve_forever, daemon=True).start()
_PROXY_BASE_URL = output.eval_js(f"google.colab.kernel.proxyPort({_port})").rstrip("/")

_paramedic_url = f"{_PROXY_BASE_URL}?k={APP_TOKEN}"
_er_dashboard_url = f"{_PROXY_BASE_URL}/er?k={APP_TOKEN}"

display(HTML(f"""
<div dir="rtl" style="max-width:860px;margin:10px auto;display:grid;grid-template-columns:1fr 1fr;gap:12px;font-family:system-ui,sans-serif;text-align:right;">
  <div style="padding:14px 16px;border:2px solid #0284c7;border-radius:12px;background:#f0f9ff;">
    <a href="{_paramedic_url}" target="_blank" rel="noopener" style="font-weight:800;font-size:15.5px;color:#0a192f;text-decoration:none;display:block;">🚑 1. RELAY — واجهة المسعف (سيارة الإسعاف)</a>
    <div style="font-size:12px;color:#334155;margin-top:4px;">Bridging the Gap Before Arrival — التسجيل الصوتي المباشر، الفرز الذكي، ومراجعة المسودة.</div>
  </div>
  <div style="padding:14px 16px;border:2px solid #0a192f;border-radius:12px;background:#ffffff;">
    <a href="{_er_dashboard_url}" target="_blank" rel="noopener" style="font-weight:800;font-size:15.5px;color:#0a192f;text-decoration:none;display:block;">🏥 2. RELAY — شاشة استقبال الطوارئ (ER Dashboard)</a>
    <div style="font-size:12px;color:#334155;margin-top:4px;">شاشة المستشفى الحية — تستقبل البلاغات لحظياً برقم المريض وصوت التنبيه.</div>
  </div>
</div>
"""))
display(HTML(_render_html()))

RELAY — Bridging the Gap Before Arrival | سلسلة النماذج المعتمدة: gemini-3.5-flash-lite ← gemini-2.5-flash-lite ← gemini-flash-lite-latest ← gemini-2.5-flash ← gemini-flash-latest ← gemini-3-flash-preview ← gemini-3.1-flash-lite-preview ← gemini-3.1-flash-lite ← gemini-3.5-flash
